# 05 GHSL Bengaluru Metropolitan Production
Uses the frozen Hyderabad-validated baseline. Run one cell at a time. Cells 3-6 are gates; do not launch Cell 7 unless all pass.


In [1]:
# CELL 1: BOOTSTRAP
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess
drive.mount('/content/drive', force_remount=False)
D=Path('/content/drive/MyDrive'); R=D/'remote_exec.sh'; K=D/'hadoop-hdfs.pem'; L=Path('/content/hadoop-hdfs.pem')
if not R.is_file(): raise FileNotFoundError(R)
os.chmod(R,0o700)
if not L.is_file(): shutil.copy2(K,L)
os.chmod(L,0o600)
def run_remote(command,timeout=300,show_output=True):
    r=subprocess.run(['bash',str(R),command],text=True,capture_output=True,timeout=timeout)
    if show_output and r.stdout: print(r.stdout)
    if show_output and r.stderr: print('REMOTE STDERR:'); print(r.stderr)
    if r.returncode: raise RuntimeError(f'Remote command failed with return code {r.returncode}.')
    return r
print('CELL 1: SUCCESS')


Mounted at /content/drive
CELL 1: SUCCESS


In [4]:
# CELL 2: FROZEN BENGALURU CONFIGURATION
ACTIVE_CITY_SLUG='bengaluru'
ACTIVE_CITY={'city_name':'Bengaluru Metropolitan Region','city_slug':'bengaluru','seed_lon':77.5946,'seed_lat':12.9716,'target_crs':'ESRI:54009','expected_grid_cells':4900,'expected_complete_cells':4900,'expected_partial_cells':0,'extent_mollweide':{'min_x':7618000.0,'min_y':1563000.0,'max_x':7688000.0,'max_y':1633000.0},'hdfs_output_base':'hdfs://localhost:9000/data/ghsl/curated/cities/city=bengaluru','remote_work_dir':'/home/ubuntu/ghsl_multicity/bengaluru'}
assert ACTIVE_CITY['extent_mollweide']['max_x']-ACTIVE_CITY['extent_mollweide']['min_x']==70000
print(ACTIVE_CITY); print('CELL 2: SUCCESS')

{'city_name': 'Bengaluru Metropolitan Region', 'city_slug': 'bengaluru', 'seed_lon': 77.5946, 'seed_lat': 12.9716, 'target_crs': 'ESRI:54009', 'expected_grid_cells': 4900, 'expected_complete_cells': 4900, 'expected_partial_cells': 0, 'extent_mollweide': {'min_x': 7618000.0, 'min_y': 1563000.0, 'max_x': 7688000.0, 'max_y': 1633000.0}, 'hdfs_output_base': 'hdfs://localhost:9000/data/ghsl/curated/cities/city=bengaluru', 'remote_work_dir': '/home/ubuntu/ghsl_multicity/bengaluru'}
CELL 2: SUCCESS


In [5]:
# ============================================================
# CELL 3: BASELINE AND HDFS PREFLIGHT
#
# Corrected behavior:
#   - Verifies the frozen SHA-256 inventory
#   - Compiles baseline Python files into /tmp
#   - Does not create __pycache__ in the baseline
#   - Removes bytecode created by the previous Cell 3 attempt
#   - Verifies all permanent baseline files are read-only
#
# Does not:
#   - modify baseline source files
#   - launch Spark
#   - modify HDFS
# ============================================================

cmd = r"""
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

BASE="/home/ubuntu/ghsl_multicity_baseline"
COMPILE_ROOT="/tmp/ghsl_baseline_compile_check"

echo "============================================================"
echo "BENGALURU BASELINE AND HDFS PREFLIGHT"
echo "============================================================"


# ------------------------------------------------------------
# HDFS service validation.
# ------------------------------------------------------------

for daemon in \
    NameNode \
    DataNode \
    SecondaryNameNode

do
    jps \
        | awk '{print $2}' \
        | grep -qx "$daemon"
done

hdfs dfsadmin -safemode get \
    | grep -qi "Safe mode is OFF"

echo "HDFS services: PASS"


# ------------------------------------------------------------
# Required baseline files.
# ------------------------------------------------------------

required_files=(
    "$BASE/baseline_manifest.json"
    "$BASE/SHA256SUMS"
    "$BASE/README.txt"
    "$BASE/source_city_config.json"
    "$BASE/batch/run_city_epochs.py"
    "$BASE/templates/population_epoch_template.py"
    "$BASE/templates/smod_epoch_template.py"
    "$BASE/templates/built_surface_epoch_template.py"
)

for file in "${required_files[@]}"; do
    if [ ! -f "$file" ]; then
        echo "ERROR: Required baseline file missing:"
        echo "$file"
        exit 20
    fi
done

echo "Required baseline files: PASS"


# ------------------------------------------------------------
# Verify the permanent baseline inventory before doing any
# compilation.
# ------------------------------------------------------------

cd "$BASE"

sha256sum -c SHA256SUMS

echo "Baseline SHA-256 inventory: PASS"


# ------------------------------------------------------------
# Remove only generated Python bytecode created by the earlier
# preflight attempt.
#
# No source, manifest, checksum, config, or README is removed.
# ------------------------------------------------------------

find "$BASE" \
    -type d \
    -name "__pycache__" \
    -print \
    -exec rm -rf {} + \
    2>/dev/null \
    || true

find "$BASE" \
    -type f \
    \( -name "*.pyc" -o -name "*.pyo" \) \
    -print \
    -delete \
    2>/dev/null \
    || true

echo "Generated baseline bytecode removed: PASS"


# ------------------------------------------------------------
# Compile outside the frozen baseline.
# ------------------------------------------------------------

rm -rf "$COMPILE_ROOT"

mkdir -p \
    "$COMPILE_ROOT/batch" \
    "$COMPILE_ROOT/templates"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile \
    "$BASE/batch/run_city_epochs.py"

for file in \
    "$BASE/templates/population_epoch_template.py" \
    "$BASE/templates/smod_epoch_template.py" \
    "$BASE/templates/built_surface_epoch_template.py"

do
    PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
    python3 -m py_compile "$file"
done

echo "Baseline Python compilation: PASS"


# ------------------------------------------------------------
# Confirm compilation created nothing inside the baseline.
# ------------------------------------------------------------

if find "$BASE" \
    -type d \
    -name "__pycache__" \
    | grep -q .
then
    echo "ERROR: Compilation created __pycache__ in baseline."
    exit 21
fi

if find "$BASE" \
    -type f \
    \( -name "*.pyc" -o -name "*.pyo" \) \
    | grep -q .
then
    echo "ERROR: Compilation created bytecode in baseline."
    exit 22
fi

echo "Baseline remains bytecode-free: PASS"


# ------------------------------------------------------------
# Verify all permanent baseline files are read-only.
#
# The baseline directories remain owner-writable so future
# city workspaces can be created elsewhere. Only files need
# to remain frozen.
# ------------------------------------------------------------

writable_files="$(
    find "$BASE" \
        -type f \
        -perm -u=w \
        -print
)"

if [ -n "$writable_files" ]; then
    echo "ERROR: Writable permanent baseline files found:"
    echo "$writable_files"
    exit 23
fi

echo "Baseline permanent files read-only: PASS"


# ------------------------------------------------------------
# Recheck the inventory after cleanup and compilation.
# ------------------------------------------------------------

cd "$BASE"

sha256sum -c SHA256SUMS

echo "Post-check SHA-256 inventory: PASS"


# ------------------------------------------------------------
# Clean temporary compilation output.
# ------------------------------------------------------------

rm -rf "$COMPILE_ROOT"

echo
echo "BASELINE_PREFLIGHT_SUCCESS"
"""


preflight_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 3 RESULT")
print("=" * 80)

print(
    "Return code:",
    preflight_result.returncode,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("Baseline source files modified: NO")
print("CELL 3: SUCCESS")

BENGALURU BASELINE AND HDFS PREFLIGHT
HDFS services: PASS
Required baseline files: PASS
baseline_manifest.json: OK
batch/run_city_epochs.py: OK
source_city_config.json: OK
templates/built_surface_epoch_template.py: OK
templates/population_epoch_template.py: OK
templates/smod_epoch_template.py: OK
Baseline SHA-256 inventory: PASS
Generated baseline bytecode removed: PASS
Baseline Python compilation: PASS
Baseline remains bytecode-free: PASS
Baseline permanent files read-only: PASS
baseline_manifest.json: OK
batch/run_city_epochs.py: OK
source_city_config.json: OK
templates/built_surface_epoch_template.py: OK
templates/population_epoch_template.py: OK
templates/smod_epoch_template.py: OK
Post-check SHA-256 inventory: PASS

BASELINE_PREFLIGHT_SUCCESS

REMOTE STDERR:


CELL 3 RESULT
Return code: 0
Spark launched: NO
HDFS modified: NO
Baseline source files modified: NO
CELL 3: SUCCESS


In [6]:
# ============================================================
# CELL 4: CREATE BENGALURU WORKSPACE FROM FROZEN BASELINE
#
# Source:
#   /home/ubuntu/ghsl_multicity_baseline
#
# Destination:
#   /home/ubuntu/ghsl_multicity/bengaluru
#
# Safety:
#   - Refuses to overwrite an existing partial workspace
#   - Removes its own partial workspace if this build fails
#   - Does not change the frozen baseline
#   - Does not change Hyderabad or legacy Bengaluru pilot
#   - Does not launch Spark
#   - Does not modify HDFS
# ============================================================

import base64
import hashlib
import json
import textwrap


# ============================================================
# VALIDATE NOTEBOOK STATE
# ============================================================

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1 and 2."
        )


if ACTIVE_CITY_SLUG != "bengaluru":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be bengaluru."
    )


expected_workspace = (
    "/home/ubuntu/ghsl_multicity/bengaluru"
)

expected_hdfs_root = (
    "hdfs://localhost:9000"
    "/data/ghsl/curated/cities/"
    "city=bengaluru"
)

expected_extent = {
    "min_x": 7618000.0,
    "min_y": 1563000.0,
    "max_x": 7688000.0,
    "max_y": 1633000.0,
}


if (
    ACTIVE_CITY["remote_work_dir"]
    != expected_workspace
):
    raise RuntimeError(
        "Unexpected Bengaluru workspace: "
        + ACTIVE_CITY["remote_work_dir"]
    )


if (
    ACTIVE_CITY["hdfs_output_base"]
    != expected_hdfs_root
):
    raise RuntimeError(
        "Unexpected Bengaluru HDFS root: "
        + ACTIVE_CITY["hdfs_output_base"]
    )


if (
    ACTIVE_CITY["city_name"]
    != "Bengaluru Metropolitan Region"
):
    raise RuntimeError(
        "Unexpected Bengaluru city name."
    )


if (
    ACTIVE_CITY["extent_mollweide"]
    != expected_extent
):
    raise RuntimeError(
        "Unexpected Bengaluru extent.\n"
        f"Expected: {expected_extent}\n"
        f"Actual:   "
        f"{ACTIVE_CITY['extent_mollweide']}"
    )


if (
    ACTIVE_CITY["expected_grid_cells"]
    != 4900
):
    raise RuntimeError(
        "Bengaluru expected_grid_cells must be 4900."
    )


# ============================================================
# SERIALIZE BENGALURU CONFIGURATION
# ============================================================

config_json = (
    json.dumps(
        ACTIVE_CITY,
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

config_b64 = base64.b64encode(
    config_json.encode("utf-8")
).decode("ascii")


# ============================================================
# BUILD REMOTE WORKSPACE CREATOR
# ============================================================

remote_builder = textwrap.dedent(
    r'''
    from pathlib import Path
    import ast
    import hashlib
    import json
    import py_compile
    import shutil
    import sys
    import time


    BASELINE = Path(
        "/home/ubuntu/ghsl_multicity_baseline"
    )

    TARGET = Path(
        "/home/ubuntu/ghsl_multicity/bengaluru"
    )

    CONFIG_PATH = Path(
        sys.argv[1]
    )

    TEMPLATE_NAMES = [
        "population_epoch_template.py",
        "smod_epoch_template.py",
        "built_surface_epoch_template.py",
    ]


    def sha256_file(path):
        digest = hashlib.sha256()

        with path.open("rb") as handle:
            for block in iter(
                lambda: handle.read(
                    1024 * 1024
                ),
                b"",
            ):
                digest.update(block)

        return digest.hexdigest()


    def compile_source(source, path):
        compile(
            source,
            str(path),
            "exec",
        )


    # ========================================================
    # LOAD CONFIGURATIONS
    # ========================================================

    if not CONFIG_PATH.is_file():
        raise FileNotFoundError(
            CONFIG_PATH
        )

    config = json.loads(
        CONFIG_PATH.read_text(
            encoding="utf-8"
        )
    )

    source_config_path = (
        BASELINE
        / "source_city_config.json"
    )

    if not source_config_path.is_file():
        raise FileNotFoundError(
            source_config_path
        )

    source_config = json.loads(
        source_config_path.read_text(
            encoding="utf-8"
        )
    )


    # ========================================================
    # VALIDATE BENGALURU CONFIGURATION
    # ========================================================

    expected_extent = {
        "min_x": 7618000.0,
        "min_y": 1563000.0,
        "max_x": 7688000.0,
        "max_y": 1633000.0,
    }

    if (
        config.get("city_slug")
        != "bengaluru"
    ):
        raise RuntimeError(
            "Bengaluru configuration has the wrong slug."
        )

    if (
        config.get("city_name")
        != "Bengaluru Metropolitan Region"
    ):
        raise RuntimeError(
            "Bengaluru configuration has the wrong name."
        )

    if (
        config.get("remote_work_dir")
        != str(TARGET)
    ):
        raise RuntimeError(
            "Bengaluru configuration has the wrong "
            "remote workspace."
        )

    if (
        config.get("hdfs_output_base")
        != (
            "hdfs://localhost:9000"
            "/data/ghsl/curated/cities/"
            "city=bengaluru"
        )
    ):
        raise RuntimeError(
            "Bengaluru configuration has the wrong "
            "HDFS output root."
        )

    if (
        config.get("expected_grid_cells")
        != 4900
    ):
        raise RuntimeError(
            "Bengaluru expected_grid_cells must be 4900."
        )

    if (
        config.get("extent_mollweide")
        != expected_extent
    ):
        raise RuntimeError(
            "Bengaluru configuration has the wrong extent."
        )


    # ========================================================
    # VALIDATE BASELINE INPUTS
    # ========================================================

    baseline_driver = (
        BASELINE
        / "batch"
        / "run_city_epochs.py"
    )

    baseline_manifest = (
        BASELINE
        / "baseline_manifest.json"
    )

    baseline_checksums = (
        BASELINE
        / "SHA256SUMS"
    )

    required_baseline_files = [
        baseline_driver,
        baseline_manifest,
        baseline_checksums,
        source_config_path,
    ]

    for required_path in (
        required_baseline_files
    ):
        if not required_path.is_file():
            raise FileNotFoundError(
                required_path
            )

    baseline_templates = {}

    for template_name in TEMPLATE_NAMES:
        template_path = (
            BASELINE
            / "templates"
            / template_name
        )

        if not template_path.is_file():
            raise FileNotFoundError(
                template_path
            )

        baseline_templates[
            template_name
        ] = template_path


    # ========================================================
    # CHECK FOR AN EXISTING WORKSPACE
    # ========================================================

    workspace_manifest = (
        TARGET
        / "bengaluru_workspace_manifest.json"
    )

    if TARGET.exists():
        if workspace_manifest.is_file():
            print("=" * 80)
            print("BENGALURU WORKSPACE CREATION")
            print("=" * 80)

            print(
                "Existing completed Bengaluru workspace "
                "found."
            )

            print(
                "Workspace:",
                TARGET,
            )

            print(
                "Existing workspace was not "
                "overwritten."
            )

            print(
                "BENGALURU_WORKSPACE_ALREADY_EXISTS"
            )

            print(
                "BENGALURU_WORKSPACE_BUILD_SUCCESS"
            )

            raise SystemExit(0)

        raise RuntimeError(
            "An incomplete Bengaluru workspace already "
            "exists and was not overwritten: "
            + str(TARGET)
        )


    # ========================================================
    # BUILD PARAMETER REPLACEMENTS
    # ========================================================

    source_root = str(
        source_config[
            "remote_work_dir"
        ]
    )

    target_root = str(
        config[
            "remote_work_dir"
        ]
    )

    source_slug = str(
        source_config[
            "city_slug"
        ]
    )

    target_slug = str(
        config[
            "city_slug"
        ]
    )

    source_name = str(
        source_config[
            "city_name"
        ]
    )

    target_name = str(
        config[
            "city_name"
        ]
    )

    source_extent = (
        source_config[
            "extent_mollweide"
        ]
    )

    target_extent = (
        config[
            "extent_mollweide"
        ]
    )

    replacements = [
        (
            source_root,
            target_root,
        ),
        (
            "city=" + source_slug,
            "city=" + target_slug,
        ),
        (
            source_slug,
            target_slug,
        ),
        (
            source_name,
            target_name,
        ),
    ]

    for key in [
        "min_x",
        "min_y",
        "max_x",
        "max_y",
    ]:
        replacements.append(
            (
                str(
                    float(
                        source_extent[key]
                    )
                ),
                str(
                    float(
                        target_extent[key]
                    )
                ),
            )
        )

        replacements.append(
            (
                str(
                    int(
                        source_extent[key]
                    )
                ),
                str(
                    int(
                        target_extent[key]
                    )
                ),
            )
        )


    def parameterize(source):
        output = source

        for old_value, new_value in replacements:
            output = output.replace(
                old_value,
                new_value,
            )

        return output


    # ========================================================
    # CREATE WORKSPACE
    # ========================================================

    try:
        (
            TARGET
            / "templates"
        ).mkdir(
            parents=True,
            exist_ok=False,
        )

        (
            TARGET
            / "programs"
        ).mkdir()

        (
            TARGET
            / "batch"
        ).mkdir()


        # ====================================================
        # PARAMETERIZE TEMPLATES
        # ====================================================

        template_hashes = {}

        for template_name in TEMPLATE_NAMES:
            source_path = (
                baseline_templates[
                    template_name
                ]
            )

            destination_path = (
                TARGET
                / "templates"
                / template_name
            )

            template_source = (
                source_path.read_text(
                    encoding="utf-8"
                )
            )

            template_source = parameterize(
                template_source
            )

            compile_source(
                template_source,
                destination_path,
            )

            destination_path.write_text(
                template_source,
                encoding="utf-8",
            )

            py_compile.compile(
                str(destination_path),
                doraise=True,
            )

            template_hashes[
                template_name
            ] = sha256_file(
                destination_path
            )


        # ====================================================
        # PARAMETERIZE DRIVER
        # ====================================================

        driver_destination = (
            TARGET
            / "batch"
            / "run_city_epochs.py"
        )

        driver_source = (
            baseline_driver.read_text(
                encoding="utf-8"
            )
        )

        driver_source = parameterize(
            driver_source
        )


        # ====================================================
        # REFRESH TEMPLATE CHECKSUMS THROUGH AST
        # ====================================================

        driver_tree = ast.parse(
            driver_source
        )

        checksum_node = None

        for node in driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                checksum_node = node
                break

        if checksum_node is None:
            raise RuntimeError(
                "EXPECTED_TEMPLATE_SHA256 "
                "was not found."
            )

        new_checksum_dictionary = {
            "population": (
                template_hashes[
                    "population_epoch_template.py"
                ]
            ),
            "smod": (
                template_hashes[
                    "smod_epoch_template.py"
                ]
            ),
            "built_surface": (
                template_hashes[
                    "built_surface_epoch_template.py"
                ]
            ),
        }

        driver_lines = (
            driver_source.splitlines(
                keepends=True
            )
        )

        checksum_assignment = (
            "EXPECTED_TEMPLATE_SHA256 = "
            + repr(
                new_checksum_dictionary
            )
            + chr(10)
        )

        driver_lines[
            checksum_node.lineno - 1:
            checksum_node.end_lineno
        ] = [
            checksum_assignment
        ]

        driver_source = "".join(
            driver_lines
        )

        compile_source(
            driver_source,
            driver_destination,
        )

        driver_destination.write_text(
            driver_source,
            encoding="utf-8",
        )

        py_compile.compile(
            str(driver_destination),
            doraise=True,
        )


        # ====================================================
        # SAVE CONFIGURATION
        # ====================================================

        bengaluru_config_path = (
            TARGET
            / "city_config.json"
        )

        bengaluru_config_path.write_text(
            json.dumps(
                config,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


        # ====================================================
        # CODE-LEVEL PARAMETER AUDIT
        #
        # The full display name is validated in city_config,
        # not required inside driver/template source.
        # ====================================================

        combined_source = (
            driver_source
        )

        for template_name in TEMPLATE_NAMES:
            combined_source += (
                TARGET
                / "templates"
                / template_name
            ).read_text(
                encoding="utf-8"
            )

        forbidden_values = [
            source_root,
            "city=" + source_slug,
            source_name,
        ]

        remaining_forbidden = [
            value
            for value in forbidden_values
            if value in combined_source
        ]

        if remaining_forbidden:
            raise RuntimeError(
                "Source-city leakage remains: "
                + repr(
                    remaining_forbidden
                )
            )

        required_code_values = [
            target_root,
            "city=" + target_slug,
            "7618000",
            "1563000",
            "7688000",
            "1633000",
        ]

        missing_code_values = [
            value
            for value
            in required_code_values
            if value
            not in combined_source
        ]

        if missing_code_values:
            raise RuntimeError(
                "Required Bengaluru code parameters "
                "are missing: "
                + repr(
                    missing_code_values
                )
            )


        # ====================================================
        # CONFIGURATION-LEVEL AUDIT
        # ====================================================

        saved_config = json.loads(
            bengaluru_config_path.read_text(
                encoding="utf-8"
            )
        )

        if (
            saved_config["city_name"]
            != "Bengaluru Metropolitan Region"
        ):
            raise RuntimeError(
                "Bengaluru city name is incorrect "
                "in city_config.json."
            )

        if (
            saved_config["city_slug"]
            != "bengaluru"
        ):
            raise RuntimeError(
                "Bengaluru city slug is incorrect "
                "in city_config.json."
            )

        if (
            saved_config["extent_mollweide"]
            != expected_extent
        ):
            raise RuntimeError(
                "Bengaluru extent is incorrect "
                "in city_config.json."
            )


        # ====================================================
        # VERIFY CHECKSUM DICTIONARY FROM SAVED DRIVER
        # ====================================================

        saved_driver_tree = ast.parse(
            driver_destination.read_text(
                encoding="utf-8"
            )
        )

        saved_checksum_dictionary = None

        for node in saved_driver_tree.body:
            if not isinstance(
                node,
                ast.Assign,
            ):
                continue

            if any(
                isinstance(
                    target,
                    ast.Name,
                )
                and target.id
                == "EXPECTED_TEMPLATE_SHA256"
                for target
                in node.targets
            ):
                saved_checksum_dictionary = (
                    ast.literal_eval(
                        node.value
                    )
                )

                break

        if (
            saved_checksum_dictionary
            != new_checksum_dictionary
        ):
            raise RuntimeError(
                "Saved driver checksum dictionary "
                "does not match Bengaluru templates."
            )


        # ====================================================
        # REMOVE GENERATED BYTECODE
        # ====================================================

        for pycache in TARGET.rglob(
            "__pycache__"
        ):
            shutil.rmtree(
                pycache
            )


        # ====================================================
        # SAVE WORKSPACE MANIFEST
        # ====================================================

        manifest = {
            "city_slug": target_slug,
            "city_name": target_name,
            "source_baseline": str(
                BASELINE
            ),
            "workspace": str(
                TARGET
            ),
            "driver_sha256": sha256_file(
                driver_destination
            ),
            "template_sha256": (
                new_checksum_dictionary
            ),
            "extent_mollweide": (
                expected_extent
            ),
            "expected_grid_cells": 4900,
            "created_at_utc": (
                time.strftime(
                    "%Y-%m-%dT%H:%M:%SZ",
                    time.gmtime(),
                )
            ),
        }

        workspace_manifest.write_text(
            json.dumps(
                manifest,
                indent=2,
                sort_keys=True,
            )
            + chr(10),
            encoding="utf-8",
        )


    except Exception:
        # Do not leave behind a partial Bengaluru workspace.
        if TARGET.exists():
            shutil.rmtree(
                TARGET
            )

        raise


    # ========================================================
    # FINAL REPORT
    # ========================================================

    print("=" * 80)
    print("BENGALURU WORKSPACE CREATION")
    print("=" * 80)

    print(
        "Workspace:",
        TARGET,
    )

    print(
        "Driver:",
        TARGET
        / "batch"
        / "run_city_epochs.py",
    )

    print(
        "Configuration:",
        TARGET
        / "city_config.json",
    )

    print(
        "Manifest:",
        workspace_manifest,
    )

    print()
    print("Template parameterization: PASS")
    print("Template compilation: PASS")
    print("Driver parameterization: PASS")
    print("Driver compilation: PASS")
    print("Template checksum refresh: PASS")
    print("Saved checksum verification: PASS")
    print("Source-city leakage audit: PASS")
    print("Bengaluru code-parameter audit: PASS")
    print("Bengaluru configuration audit: PASS")
    print("Bengaluru extent validation: PASS")
    print("Partial-build cleanup protection: PASS")
    print("Spark launched: NO")
    print("HDFS modified: NO")
    print("BENGALURU_WORKSPACE_BUILD_SUCCESS")
    '''
).strip() + "\n"


# ============================================================
# LOCAL BUILDER VALIDATION
# ============================================================

compile(
    remote_builder,
    "build_bengaluru_workspace.py",
    "exec",
)

builder_sha256 = hashlib.sha256(
    remote_builder.encode("utf-8")
).hexdigest()

print("LOCAL BENGALURU BUILDER COMPILE: PASS")

print(
    "Expected builder SHA-256:",
    builder_sha256,
)


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

builder_b64 = base64.b64encode(
    remote_builder.encode("utf-8")
).decode("ascii")


remote_command = f"""
set -e

BUILDER="/tmp/build_bengaluru_workspace.py"
CONFIG="/tmp/bengaluru_city_config.json"

EXPECTED_BUILDER_SHA256="{builder_sha256}"

echo "{builder_b64}" \
| base64 -d \
> "${{BUILDER}}"

echo "{config_b64}" \
| base64 -d \
> "${{CONFIG}}"

ACTUAL_BUILDER_SHA256="$(
    sha256sum "${{BUILDER}}" \
    | awk '{{print $1}}'
)"

echo "Remote builder SHA-256: ${{ACTUAL_BUILDER_SHA256}}"

if [ "${{ACTUAL_BUILDER_SHA256}}" != "${{EXPECTED_BUILDER_SHA256}}" ]; then
    echo "ERROR: Builder checksum mismatch."
    exit 20
fi

echo "REMOTE BUILDER CHECKSUM: PASS"

python3 -m py_compile "${{BUILDER}}"

echo "REMOTE BUILDER COMPILE: PASS"

python3 "${{BUILDER}}" "${{CONFIG}}"
"""


build_result = run_remote(
    remote_command,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 4 RESULT")
print("=" * 80)

print(
    "Return code:",
    build_result.returncode,
)

print(
    "Workspace:",
    ACTIVE_CITY["remote_work_dir"],
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("CELL 4: SUCCESS")

LOCAL BENGALURU BUILDER COMPILE: PASS
Expected builder SHA-256: 89df871be534211e3e6ec659d171ef6054d5691ce9c745eeab241e7b1bb727f2
Remote builder SHA-256: 89df871be534211e3e6ec659d171ef6054d5691ce9c745eeab241e7b1bb727f2
REMOTE BUILDER CHECKSUM: PASS
REMOTE BUILDER COMPILE: PASS
BENGALURU WORKSPACE CREATION
Workspace: /home/ubuntu/ghsl_multicity/bengaluru
Driver: /home/ubuntu/ghsl_multicity/bengaluru/batch/run_city_epochs.py
Configuration: /home/ubuntu/ghsl_multicity/bengaluru/city_config.json
Manifest: /home/ubuntu/ghsl_multicity/bengaluru/bengaluru_workspace_manifest.json

Template parameterization: PASS
Template compilation: PASS
Driver parameterization: PASS
Driver compilation: PASS
Template checksum refresh: PASS
Saved checksum verification: PASS
Source-city leakage audit: PASS
Bengaluru code-parameter audit: PASS
Bengaluru configuration audit: PASS
Bengaluru extent validation: PASS
Partial-build cleanup protection: PASS
Spark launched: NO
HDFS modified: NO
BENGALURU_WORKSPACE_BUILD_

In [7]:
# ============================================================
# CELL 4A: BENGALURU 2025/2030 SOURCE-CATALOG PREFLIGHT
#
# Checks the local HDFS GeoTIFF catalog for:
#   - population 2025 and 2030
#   - SMOD 2025 and 2030
#   - built_surface 2025 and 2030
#
# For each product and epoch, reports:
#   - total catalog records
#   - records with usable path and bounds
#   - source shards intersecting Bengaluru's 70 × 70 km extent
#   - filenames and paths of intersecting shards
#   - source width, height, bands and resolution
#
# Read-only:
#   - Does not modify the Bengaluru workspace
#   - Does not modify the source catalog
#   - Does not modify HDFS
#   - Does not create production outputs
#
# A temporary Spark program is created under /tmp and removed.
# ============================================================

import base64
import hashlib
import json
import textwrap


# ============================================================
# VALIDATE NOTEBOOK STATE
# ============================================================

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1 and 2."
        )


if ACTIVE_CITY_SLUG != "bengaluru":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be bengaluru."
    )


expected_extent = {
    "min_x": 7618000.0,
    "min_y": 1563000.0,
    "max_x": 7688000.0,
    "max_y": 1633000.0,
}


if (
    ACTIVE_CITY[
        "extent_mollweide"
    ]
    != expected_extent
):
    raise RuntimeError(
        "Unexpected Bengaluru extent.\n"
        "Expected: "
        + repr(expected_extent)
        + "\nActual:   "
        + repr(
            ACTIVE_CITY[
                "extent_mollweide"
            ]
        )
    )


protected_epochs = [
    2025,
    2030,
]


required_products = [
    "population",
    "smod",
    "built_surface",
]


catalog_path = (
    "hdfs://localhost:9000"
    "/data/ghsl/metadata/india/"
    "geotiff_catalog"
)


audit_config = {
    "catalog_path": catalog_path,
    "city_slug": (
        ACTIVE_CITY_SLUG
    ),
    "extent_mollweide": (
        expected_extent
    ),
    "required_products": (
        required_products
    ),
    "protected_epochs": (
        protected_epochs
    ),
}


config_json = json.dumps(
    audit_config,
    sort_keys=True,
)


config_b64 = base64.b64encode(
    config_json.encode("utf-8")
).decode("ascii")


# ============================================================
# REMOTE SPARK AUDIT PROGRAM
# ============================================================

remote_audit_program = textwrap.dedent(
    r'''
    from pyspark.sql import (
        SparkSession,
        functions as F,
    )

    from sedona.spark import (
        SedonaContext,
    )

    from pathlib import Path

    import json
    import sys


    CONFIG_PATH = Path(
        sys.argv[1]
    )


    if not CONFIG_PATH.is_file():
        raise FileNotFoundError(
            CONFIG_PATH
        )


    config = json.loads(
        CONFIG_PATH.read_text(
            encoding="utf-8"
        )
    )


    CATALOG_PATH = config[
        "catalog_path"
    ]

    CITY_SLUG = config[
        "city_slug"
    ]

    EXTENT = config[
        "extent_mollweide"
    ]

    REQUIRED_PRODUCTS = config[
        "required_products"
    ]

    PROTECTED_EPOCHS = [
        int(epoch)
        for epoch
        in config[
            "protected_epochs"
        ]
    ]


    # ========================================================
    # CREATE SPARK + SEDONA CONTEXT
    # ========================================================

    spark_config = (
        SedonaContext.builder()
        .appName(
            "GHSL-Bengaluru-"
            "Protected-Epoch-Catalog-Audit"
        )
        .getOrCreate()
    )

    spark = SedonaContext.create(
        spark_config
    )

    spark.sparkContext.setLogLevel(
        "WARN"
    )


    try:
        # ====================================================
        # READ AND VALIDATE CATALOG
        # ====================================================

        catalog_df = (
            spark.read
            .parquet(
                CATALOG_PATH
            )
        )


        catalog_columns = set(
            catalog_df.columns
        )


        required_catalog_columns = {
            "product",
            "temporal_value",
            "filename",
            "path",
            "fileSize",
            "width",
            "height",
            "numBands",
            "resolution_x",
            "resolution_y",
            "min_x",
            "min_y",
            "max_x",
            "max_y",
        }


        missing_catalog_columns = sorted(
            required_catalog_columns
            - catalog_columns
        )


        if missing_catalog_columns:
            raise RuntimeError(
                "The GeoTIFF catalog is missing "
                "required columns: "
                + repr(
                    missing_catalog_columns
                )
            )


        total_catalog_records = (
            catalog_df.count()
        )


        if total_catalog_records <= 0:
            raise RuntimeError(
                "The GeoTIFF catalog is empty."
            )


        catalog_df.createOrReplaceTempView(
            "ghsl_geotiff_catalog"
        )


        # ====================================================
        # DISCOVER PRODUCT AND EPOCH VALUES
        # ====================================================

        discovered_product_epoch_rows = (
            catalog_df
            .select(
                F.lower(
                    F.trim(
                        F.col("product")
                    )
                ).alias(
                    "product"
                ),
                F.col(
                    "temporal_value"
                )
                .cast("int")
                .alias(
                    "epoch"
                ),
            )
            .filter(
                F.col(
                    "product"
                ).isNotNull()
            )
            .filter(
                F.col(
                    "epoch"
                ).isNotNull()
            )
            .distinct()
            .orderBy(
                "product",
                "epoch",
            )
            .collect()
        )


        discovered_product_epochs = {}


        for row in (
            discovered_product_epoch_rows
        ):
            product = row[
                "product"
            ]

            epoch = int(
                row[
                    "epoch"
                ]
            )

            discovered_product_epochs.setdefault(
                product,
                [],
            ).append(
                epoch
            )


        # ====================================================
        # AUDIT REQUIRED PRODUCT-EPOCH COMBINATIONS
        # ====================================================

        reports = []


        for product in REQUIRED_PRODUCTS:
            for epoch in PROTECTED_EPOCHS:
                product_epoch_df = (
                    catalog_df
                    .filter(
                        F.lower(
                            F.trim(
                                F.col(
                                    "product"
                                )
                            )
                        )
                        == F.lit(
                            product
                        )
                    )
                    .filter(
                        F.col(
                            "temporal_value"
                        )
                        .cast("int")
                        == F.lit(
                            epoch
                        )
                    )
                )


                catalog_record_count = (
                    product_epoch_df.count()
                )


                usable_catalog_df = (
                    product_epoch_df
                    .filter(
                        F.col(
                            "path"
                        ).isNotNull()
                    )
                    .filter(
                        F.col(
                            "min_x"
                        ).isNotNull()
                    )
                    .filter(
                        F.col(
                            "min_y"
                        ).isNotNull()
                    )
                    .filter(
                        F.col(
                            "max_x"
                        ).isNotNull()
                    )
                    .filter(
                        F.col(
                            "max_y"
                        ).isNotNull()
                    )
                )


                usable_catalog_count = (
                    usable_catalog_df.count()
                )


                usable_catalog_df.createOrReplaceTempView(
                    "required_product_epoch"
                )


                intersecting_df = spark.sql(
                    f"""
                    SELECT
                        filename,
                        path,
                        fileSize,
                        width,
                        height,
                        numBands,
                        resolution_x,
                        resolution_y,
                        min_x,
                        min_y,
                        max_x,
                        max_y

                    FROM required_product_epoch

                    WHERE ST_Intersects(
                        ST_PolygonFromEnvelope(
                            min_x,
                            min_y,
                            max_x,
                            max_y
                        ),
                        ST_PolygonFromEnvelope(
                            {float(EXTENT["min_x"])},
                            {float(EXTENT["min_y"])},
                            {float(EXTENT["max_x"])},
                            {float(EXTENT["max_y"])}
                        )
                    )

                    ORDER BY filename
                    """
                )


                intersecting_records = [
                    row.asDict(
                        recursive=True
                    )
                    for row
                    in intersecting_df.collect()
                ]


                intersecting_count = len(
                    intersecting_records
                )


                distinct_intersecting_paths = len(
                    {
                        record[
                            "path"
                        ]
                        for record
                        in intersecting_records
                        if record[
                            "path"
                        ]
                        is not None
                    }
                )


                available = (
                    catalog_record_count
                    > 0
                )


                usable = (
                    usable_catalog_count
                    > 0
                )


                intersects_city = (
                    intersecting_count
                    > 0
                )


                report = {
                    "product": product,
                    "epoch": epoch,
                    "catalog_record_count": (
                        catalog_record_count
                    ),
                    "usable_catalog_count": (
                        usable_catalog_count
                    ),
                    "intersecting_shard_count": (
                        intersecting_count
                    ),
                    "distinct_intersecting_paths": (
                        distinct_intersecting_paths
                    ),
                    "available": available,
                    "usable": usable,
                    "intersects_city": (
                        intersects_city
                    ),
                    "intersecting_records": (
                        intersecting_records
                    ),
                }


                reports.append(
                    report
                )


                print(
                    "AUDIT:",
                    product,
                    epoch,
                    "catalog_records="
                    + str(
                        catalog_record_count
                    ),
                    "usable_records="
                    + str(
                        usable_catalog_count
                    ),
                    "intersecting_shards="
                    + str(
                        intersecting_count
                    ),
                )


        # ====================================================
        # BUILD OUTPUT
        # ====================================================

        missing_combinations = [
            {
                "product": report[
                    "product"
                ],
                "epoch": report[
                    "epoch"
                ],
                "catalog_record_count": (
                    report[
                        "catalog_record_count"
                    ]
                ),
                "intersecting_shard_count": (
                    report[
                        "intersecting_shard_count"
                    ]
                ),
            }
            for report
            in reports
            if (
                not report[
                    "available"
                ]
                or not report[
                    "usable"
                ]
                or not report[
                    "intersects_city"
                ]
            )
        ]


        output = {
            "city_slug": (
                CITY_SLUG
            ),
            "catalog_path": (
                CATALOG_PATH
            ),
            "total_catalog_records": (
                total_catalog_records
            ),
            "extent_mollweide": (
                EXTENT
            ),
            "required_products": (
                REQUIRED_PRODUCTS
            ),
            "protected_epochs": (
                PROTECTED_EPOCHS
            ),
            "discovered_product_epochs": (
                discovered_product_epochs
            ),
            "reports": reports,
            "missing_combinations": (
                missing_combinations
            ),
            "all_required_combinations_available": (
                len(
                    missing_combinations
                )
                == 0
            ),
        }


        print(
            "CATALOG_AUDIT_JSON_BEGIN"
        )

        print(
            json.dumps(
                output,
                sort_keys=True,
            )
        )

        print(
            "CATALOG_AUDIT_JSON_END"
        )

        print(
            "BENGALURU_PROTECTED_EPOCH_"
            "CATALOG_AUDIT_COMPLETE"
        )


    finally:
        spark.stop()
    '''
).strip() + "\n"


# ============================================================
# VALIDATE AND ENCODE REMOTE AUDIT PROGRAM
# ============================================================

compile(
    remote_audit_program,
    "audit_bengaluru_2025_2030_catalog.py",
    "exec",
)


audit_program_sha256 = hashlib.sha256(
    remote_audit_program.encode(
        "utf-8"
    )
).hexdigest()


audit_program_b64 = base64.b64encode(
    remote_audit_program.encode(
        "utf-8"
    )
).decode("ascii")


print(
    "LOCAL BENGALURU CATALOG AUDIT "
    "COMPILE: PASS"
)

print(
    "Expected audit SHA-256:",
    audit_program_sha256,
)


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

remote_command = f"""
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

SCRIPT="/tmp/audit_bengaluru_2025_2030_catalog.py"
CONFIG="/tmp/bengaluru_2025_2030_catalog_config.json"

EXPECTED_SCRIPT_SHA256="{audit_program_sha256}"

echo "{audit_program_b64}" \
| base64 -d \
> "${{SCRIPT}}"

echo "{config_b64}" \
| base64 -d \
> "${{CONFIG}}"

ACTUAL_SCRIPT_SHA256="$(
    sha256sum "${{SCRIPT}}" \
    | awk '{{print $1}}'
)"

echo "Remote audit SHA-256: ${{ACTUAL_SCRIPT_SHA256}}"

if [ "${{ACTUAL_SCRIPT_SHA256}}" != "${{EXPECTED_SCRIPT_SHA256}}" ]; then
    echo "ERROR: Audit-script checksum mismatch."
    exit 20
fi

echo "REMOTE AUDIT CHECKSUM: PASS"

python3 -m py_compile "${{SCRIPT}}"

echo "REMOTE AUDIT COMPILE: PASS"

spark-submit \
    --master local[1] \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=2 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    --packages \
org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,org.datasyslab:geotools-wrapper:1.9.1-33.5 \
    --repositories \
https://artifacts.unidata.ucar.edu/repository/unidata-all \
    "${{SCRIPT}}" \
    "${{CONFIG}}"

rm -f \
    "${{SCRIPT}}" \
    "${{CONFIG}}"

echo "Temporary audit files removed: PASS"
echo "Bengaluru workspace modified: NO"
echo "HDFS modified: NO"
echo "BENGALURU_CATALOG_PREFLIGHT_REMOTE_SUCCESS"
"""


catalog_audit_result = run_remote(
    remote_command,
    timeout=1200,
    show_output=False,
)


# ============================================================
# PARSE REMOTE JSON WITHOUT A MULTILINE REGEX
# ============================================================

begin_marker = (
    "CATALOG_AUDIT_JSON_BEGIN"
)

end_marker = (
    "CATALOG_AUDIT_JSON_END"
)


begin_index = (
    catalog_audit_result.stdout.find(
        begin_marker
    )
)

end_index = (
    catalog_audit_result.stdout.find(
        end_marker
    )
)


if (
    begin_index < 0
    or end_index < 0
    or end_index <= begin_index
):
    print(
        catalog_audit_result.stdout[
            -12000:
        ]
    )

    if catalog_audit_result.stderr:
        print(
            "REMOTE STDERR:"
        )

        print(
            catalog_audit_result.stderr[
                -4000:
            ]
        )

    raise RuntimeError(
        "Could not locate the remote "
        "catalog-audit JSON markers."
    )


json_start = (
    begin_index
    + len(
        begin_marker
    )
)


catalog_audit_json = (
    catalog_audit_result.stdout[
        json_start:
        end_index
    ]
    .strip()
)


if not catalog_audit_json:
    raise RuntimeError(
        "The catalog-audit JSON payload is empty."
    )


try:
    BENGALURU_PROTECTED_EPOCH_CATALOG_AUDIT = (
        json.loads(
            catalog_audit_json
        )
    )

except json.JSONDecodeError as error:
    print(
        catalog_audit_json[
            :12000
        ]
    )

    raise RuntimeError(
        "Could not parse the Bengaluru "
        "catalog-audit JSON."
    ) from error


# ============================================================
# VALIDATE EXPECTED REPORT STRUCTURE
# ============================================================

reports = (
    BENGALURU_PROTECTED_EPOCH_CATALOG_AUDIT.get(
        "reports",
        [],
    )
)


expected_report_count = (
    len(required_products)
    * len(protected_epochs)
)


if (
    len(reports)
    != expected_report_count
):
    raise RuntimeError(
        "Expected "
        + str(
            expected_report_count
        )
        + " product-epoch reports, found "
        + str(
            len(reports)
        )
    )


report_lookup = {
    (
        report["product"],
        int(
            report["epoch"]
        ),
    ): report
    for report
    in reports
}


# ============================================================
# REPORT
# ============================================================

print()
print("=" * 80)
print(
    "BENGALURU 2025/2030 SOURCE-CATALOG PREFLIGHT"
)
print("=" * 80)

print(
    "Catalog:",
    BENGALURU_PROTECTED_EPOCH_CATALOG_AUDIT[
        "catalog_path"
    ],
)

print(
    "Catalog records:",
    BENGALURU_PROTECTED_EPOCH_CATALOG_AUDIT[
        "total_catalog_records"
    ],
)

print(
    "Bengaluru extent:",
    BENGALURU_PROTECTED_EPOCH_CATALOG_AUDIT[
        "extent_mollweide"
    ],
)


for product in required_products:
    print()

    print(
        product.upper()
    )

    for epoch in protected_epochs:
        report = report_lookup[
            (
                product,
                epoch,
            )
        ]

        if (
            report["available"]
            and report["usable"]
            and report["intersects_city"]
        ):
            status = "AVAILABLE"
        elif not report["available"]:
            status = "MISSING_FROM_CATALOG"
        elif not report["usable"]:
            status = "CATALOG_RECORDS_UNUSABLE"
        else:
            status = "NO_BENGALURU_INTERSECTION"

        print(
            " ",
            epoch,
            ":",
            status,
            "| catalog records:",
            report[
                "catalog_record_count"
            ],
            "| intersecting shards:",
            report[
                "intersecting_shard_count"
            ],
        )

        for source_record in (
            report[
                "intersecting_records"
            ]
        ):
            print(
                "    -",
                source_record[
                    "filename"
                ],
            )

            print(
                "      path:",
                source_record[
                    "path"
                ],
            )

            print(
                "      shape:",
                (
                    source_record[
                        "width"
                    ],
                    source_record[
                        "height"
                    ],
                    source_record[
                        "numBands"
                    ],
                ),
            )

            print(
                "      resolution:",
                (
                    source_record[
                        "resolution_x"
                    ],
                    source_record[
                        "resolution_y"
                    ],
                ),
            )


all_available = (
    BENGALURU_PROTECTED_EPOCH_CATALOG_AUDIT[
        "all_required_combinations_available"
    ]
)


print()

print(
    "All six protected product-epoch "
    "combinations available:",
    all_available,
)

print(
    "Bengaluru workspace modified: NO"
)

print(
    "HDFS modified: NO"
)

print(
    "Temporary audit files removed: PASS"
)


if not all_available:
    print()

    print(
        "Missing or unusable combinations:"
    )

    for missing in (
        BENGALURU_PROTECTED_EPOCH_CATALOG_AUDIT[
            "missing_combinations"
        ]
    ):
        print(
            " -",
            missing,
        )

    print()

    print(
        "PROTECTED_EPOCH_SOURCE_INGESTION_REQUIRED"
    )

else:
    print(
        "BENGALURU_2025_2030_SOURCE_PREFLIGHT_SUCCESS"
    )


print(
    "CELL 4A: SUCCESS"
)

LOCAL BENGALURU CATALOG AUDIT COMPILE: PASS
Expected audit SHA-256: e0337ac28f05f606211a1a9ef92b074e98058f4ac7149e68802c920c438695f6

BENGALURU 2025/2030 SOURCE-CATALOG PREFLIGHT
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog
Catalog records: 325
Bengaluru extent: {'max_x': 7688000.0, 'max_y': 1633000.0, 'min_x': 7618000.0, 'min_y': 1563000.0}

POPULATION
  2025 : MISSING_FROM_CATALOG | catalog records: 0 | intersecting shards: 0
  2030 : MISSING_FROM_CATALOG | catalog records: 0 | intersecting shards: 0

SMOD
  2025 : MISSING_FROM_CATALOG | catalog records: 0 | intersecting shards: 0
  2030 : MISSING_FROM_CATALOG | catalog records: 0 | intersecting shards: 0

BUILT_SURFACE
  2025 : MISSING_FROM_CATALOG | catalog records: 0 | intersecting shards: 0
  2030 : MISSING_FROM_CATALOG | catalog records: 0 | intersecting shards: 0

All six protected product-epoch combinations available: False
Bengaluru workspace modified: NO
HDFS modified: NO
Temporary audit files remo

In [8]:
# ============================================================
# CELL 5: BENGALURU STATIC AND LEAKAGE AUDIT
#
# This version deliberately avoids:
#   - outer f-strings around embedded Python
#   - fragile grep checks for Python spacing
#   - creating __pycache__ inside the workspace
#
# It verifies:
#   - required Bengaluru files
#   - Python compilation
#   - Bengaluru configuration
#   - driver functions
#   - 4,900-cell expectation
#   - template checksums
#   - workspace-manifest checksums
#   - Bengaluru paths and extent
#   - absence of Hyderabad leakage
#   - generalized endpoint logic
#   - generalized 2020 guards
#   - absence of stale launch state
#
# Read-only toward source files and HDFS.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "bengaluru":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be bengaluru."
    )


remote_audit = r'''
set -e

ROOT="__BENGALURU_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"
CONFIG="$ROOT/city_config.json"
MANIFEST="$ROOT/bengaluru_workspace_manifest.json"

POPULATION_TEMPLATE="$ROOT/templates/population_epoch_template.py"
SMOD_TEMPLATE="$ROOT/templates/smod_epoch_template.py"
BUILT_TEMPLATE="$ROOT/templates/built_surface_epoch_template.py"

COMPILE_ROOT="/tmp/bengaluru_static_compile_check"


echo "============================================================"
echo "BENGALURU STATIC AND LEAKAGE AUDIT"
echo "============================================================"


# ============================================================
# REQUIRED FILES
# ============================================================

for required_file in \
    "$DRIVER" \
    "$CONFIG" \
    "$MANIFEST" \
    "$POPULATION_TEMPLATE" \
    "$SMOD_TEMPLATE" \
    "$BUILT_TEMPLATE"

do
    if [ ! -f "$required_file" ]; then
        echo "ERROR: Required file is missing:"
        echo "$required_file"
        exit 20
    fi
done

echo "Required Bengaluru files: PASS"


# ============================================================
# COMPILE OUTSIDE THE WORKSPACE
# ============================================================

rm -rf "$COMPILE_ROOT"

mkdir -p "$COMPILE_ROOT"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$DRIVER"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$POPULATION_TEMPLATE"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$SMOD_TEMPLATE"

PYTHONPYCACHEPREFIX="$COMPILE_ROOT" \
python3 -m py_compile "$BUILT_TEMPLATE"

echo "Bengaluru Python compilation: PASS"


# ============================================================
# STRUCTURAL PYTHON AUDIT
#
# All paths are command-line arguments. No shell or notebook
# interpolation occurs inside this Python source.
# ============================================================

python3 - \
    "$ROOT" \
    "$DRIVER" \
    "$CONFIG" \
    "$MANIFEST" \
    "$POPULATION_TEMPLATE" \
    "$SMOD_TEMPLATE" \
    "$BUILT_TEMPLATE" <<'PY'
from pathlib import Path
import ast
import hashlib
import json
import sys


root = Path(sys.argv[1])
driver_path = Path(sys.argv[2])
config_path = Path(sys.argv[3])
manifest_path = Path(sys.argv[4])

population_template = Path(sys.argv[5])
smod_template = Path(sys.argv[6])
built_template = Path(sys.argv[7])


def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


# ============================================================
# CONFIGURATION AUDIT
# ============================================================

config = json.loads(
    config_path.read_text(
        encoding="utf-8"
    )
)

expected_config = {
    "city_slug": "bengaluru",
    "city_name": "Bengaluru Metropolitan Region",
    "remote_work_dir":
        "/home/ubuntu/ghsl_multicity/bengaluru",
    "hdfs_output_base":
        "hdfs://localhost:9000"
        "/data/ghsl/curated/cities/"
        "city=bengaluru",
    "expected_grid_cells": 4900,
    "expected_complete_cells": 4900,
    "expected_partial_cells": 0,
    "target_crs": "ESRI:54009",
}

for key, expected_value in expected_config.items():
    actual_value = config.get(key)

    if actual_value != expected_value:
        raise RuntimeError(
            "Configuration mismatch for "
            + key
            + ": expected "
            + repr(expected_value)
            + ", found "
            + repr(actual_value)
        )


expected_extent = {
    "min_x": 7618000.0,
    "min_y": 1563000.0,
    "max_x": 7688000.0,
    "max_y": 1633000.0,
}

actual_extent = config.get(
    "extent_mollweide"
)

if actual_extent != expected_extent:
    raise RuntimeError(
        "Bengaluru extent mismatch. Expected "
        + repr(expected_extent)
        + ", found "
        + repr(actual_extent)
    )

if (
    expected_extent["max_x"]
    - expected_extent["min_x"]
    != 70000.0
):
    raise RuntimeError(
        "Bengaluru X extent is not 70 km."
    )

if (
    expected_extent["max_y"]
    - expected_extent["min_y"]
    != 70000.0
):
    raise RuntimeError(
        "Bengaluru Y extent is not 70 km."
    )

print("Bengaluru configuration: PASS")


# ============================================================
# DRIVER STRUCTURE
# ============================================================

driver_source = driver_path.read_text(
    encoding="utf-8"
)

driver_tree = ast.parse(
    driver_source
)

function_names = set()

for node in driver_tree.body:
    if isinstance(
        node,
        ast.FunctionDef,
    ):
        function_names.add(
            node.name
        )


required_functions = {
    "generate_population_program",
    "generate_smod_program",
    "generate_built_program",
    "ensure_stage",
    "verify_template",
    "replace_exact_count",
}

missing_functions = sorted(
    required_functions
    - function_names
)

if missing_functions:
    raise RuntimeError(
        "Required driver functions are missing: "
        + repr(missing_functions)
    )

print("Required production functions: PASS")


# ============================================================
# TOP-LEVEL ASSIGNMENTS
# ============================================================

assignments = {}

for node in driver_tree.body:
    if not isinstance(
        node,
        ast.Assign,
    ):
        continue

    try:
        assignment_value = ast.literal_eval(
            node.value
        )
    except Exception:
        continue

    for target in node.targets:
        if isinstance(
            target,
            ast.Name,
        ):
            assignments[
                target.id
            ] = assignment_value


# Find a structural expected row/cell constant equal to 4900.

grid_expectations = {}

for name, value in assignments.items():
    uppercase_name = name.upper()

    if "EXPECTED" not in uppercase_name:
        continue

    if (
        "ROW" not in uppercase_name
        and "CELL" not in uppercase_name
    ):
        continue

    grid_expectations[name] = value


if 4900 not in grid_expectations.values():
    raise RuntimeError(
        "No expected row/cell constant equal "
        "to 4900 was found. Candidates: "
        + repr(grid_expectations)
    )

print(
    "Expected 4,900-cell constant: PASS",
    grid_expectations,
)


# ============================================================
# TEMPLATE CHECKSUM AUDIT
# ============================================================

driver_checksums = assignments.get(
    "EXPECTED_TEMPLATE_SHA256"
)

if not isinstance(
    driver_checksums,
    dict,
):
    raise RuntimeError(
        "EXPECTED_TEMPLATE_SHA256 is missing "
        "or is not a dictionary."
    )


actual_checksums = {
    "population": sha256_file(
        population_template
    ),
    "smod": sha256_file(
        smod_template
    ),
    "built_surface": sha256_file(
        built_template
    ),
}


if driver_checksums != actual_checksums:
    raise RuntimeError(
        "Driver template checksums do not match. "
        "Driver="
        + repr(driver_checksums)
        + ", actual="
        + repr(actual_checksums)
    )

print("Driver template checksums: PASS")


# ============================================================
# WORKSPACE MANIFEST AUDIT
# ============================================================

manifest = json.loads(
    manifest_path.read_text(
        encoding="utf-8"
    )
)

if (
    manifest.get("city_slug")
    != "bengaluru"
):
    raise RuntimeError(
        "Workspace manifest has the wrong slug."
    )

if (
    manifest.get("city_name")
    != "Bengaluru Metropolitan Region"
):
    raise RuntimeError(
        "Workspace manifest has the wrong name."
    )

if (
    manifest.get("driver_sha256")
    != sha256_file(driver_path)
):
    raise RuntimeError(
        "Workspace driver checksum mismatch."
    )

if (
    manifest.get("template_sha256")
    != actual_checksums
):
    raise RuntimeError(
        "Workspace template checksum mismatch."
    )

print("Workspace manifest checksums: PASS")


# ============================================================
# CITY PATH AND LEAKAGE AUDIT
# ============================================================

combined_source = driver_source

combined_source += (
    population_template.read_text(
        encoding="utf-8"
    )
)

combined_source += (
    smod_template.read_text(
        encoding="utf-8"
    )
)

combined_source += (
    built_template.read_text(
        encoding="utf-8"
    )
)


required_bengaluru_values = [
    "/home/ubuntu/ghsl_multicity/bengaluru",
    "city=bengaluru",
    "7618000",
    "1563000",
    "7688000",
    "1633000",
]

missing_bengaluru_values = []

for value in required_bengaluru_values:
    if value not in combined_source:
        missing_bengaluru_values.append(
            value
        )

if missing_bengaluru_values:
    raise RuntimeError(
        "Required Bengaluru values are missing: "
        + repr(missing_bengaluru_values)
    )


forbidden_hyderabad_values = [
    "/home/ubuntu/ghsl_multicity/hyderabad",
    "city=hyderabad",
    "Hyderabad Metropolitan Region",
]

remaining_hyderabad_values = []

for value in forbidden_hyderabad_values:
    if value in combined_source:
        remaining_hyderabad_values.append(
            value
        )

if remaining_hyderabad_values:
    raise RuntimeError(
        "Hyderabad leakage remains: "
        + repr(
            remaining_hyderabad_values
        )
    )

print("Bengaluru paths and extent in code: PASS")
print("Hyderabad leakage audit: PASS")


# ============================================================
# GENERALIZED TEMPLATE LOGIC
# ============================================================

population_source = (
    population_template.read_text(
        encoding="utf-8"
    )
)

smod_source = (
    smod_template.read_text(
        encoding="utf-8"
    )
)

built_source = (
    built_template.read_text(
        encoding="utf-8"
    )
)


if (
    ".parquet(population_1975_path)"
    in population_source
):
    raise RuntimeError(
        "Population template still contains "
        "the obsolete 1975 self-reference."
    )

if (
    ".parquet(SMOD_1975_PATH)"
    in smod_source
):
    raise RuntimeError(
        "SMOD template still contains "
        "the obsolete 1975 self-reference."
    )

if (
    "product=population/"
    not in built_source
):
    raise RuntimeError(
        "Built template does not contain "
        "the population-grid reference."
    )

print("Generalized template logic: PASS")


# ============================================================
# GENERALIZED 2020 GUARDS
# ============================================================

def is_epoch_not_2020(node):
    return (
        isinstance(
            node,
            ast.Compare,
        )
        and isinstance(
            node.left,
            ast.Name,
        )
        and node.left.id == "epoch"
        and len(node.ops) == 1
        and isinstance(
            node.ops[0],
            ast.NotEq,
        )
        and len(
            node.comparators
        ) == 1
        and isinstance(
            node.comparators[0],
            ast.Constant,
        )
        and node.comparators[0].value
        == 2020
    )


def is_active_2020_check(node):
    return (
        isinstance(
            node,
            ast.Compare,
        )
        and isinstance(
            node.left,
            ast.Constant,
        )
        and node.left.value
        == "EPOCH = 2020"
        and len(node.ops) == 1
        and isinstance(
            node.ops[0],
            ast.In,
        )
        and len(
            node.comparators
        ) == 1
        and isinstance(
            node.comparators[0],
            ast.Name,
        )
        and node.comparators[0].id
        == "program"
    )


guards_checked = 0

for node in ast.walk(
    driver_tree
):
    if not isinstance(
        node,
        ast.If,
    ):
        continue

    messages = []

    for child in ast.walk(node):
        if (
            isinstance(
                child,
                ast.Constant,
            )
            and isinstance(
                child.value,
                str,
            )
        ):
            messages.append(
                child.value
            )

    if not any(
        "retains active 2020"
        in message
        for message in messages
    ):
        continue

    if not (
        isinstance(
            node.test,
            ast.BoolOp,
        )
        and isinstance(
            node.test.op,
            ast.And,
        )
    ):
        raise RuntimeError(
            "A retained-active-2020 guard "
            "is not generalized."
        )

    has_epoch_condition = any(
        is_epoch_not_2020(
            condition
        )
        for condition
        in node.test.values
    )

    has_program_condition = any(
        is_active_2020_check(
            condition
        )
        for condition
        in node.test.values
    )

    if not (
        has_epoch_condition
        and has_program_condition
    ):
        raise RuntimeError(
            "A 2020 guard is missing one "
            "of its required conditions."
        )

    guards_checked += 1


if guards_checked == 0:
    raise RuntimeError(
        "No generalized 2020 guards were found."
    )

print(
    "Generalized 2020 guards: PASS",
    guards_checked,
)


print()
print("BENGALURU_STRUCTURAL_AUDIT_SUCCESS")
PY


# ============================================================
# LAUNCH-STATE AUDIT
# ============================================================

if [ -e "$ROOT/batch/city.pid" ]; then
    echo "ERROR: Bengaluru city.pid already exists."
    exit 30
fi

if [ -e "$ROOT/batch/city.log" ]; then
    echo "ERROR: Bengaluru city.log already exists."
    exit 31
fi

echo "No stale Bengaluru launch state: PASS"


# ============================================================
# CLEAN TEMPORARY COMPILATION OUTPUT
# ============================================================

rm -rf "$COMPILE_ROOT"


echo
echo "BENGALURU_STATIC_AUDIT_SUCCESS"
'''


# Replace a literal placeholder instead of using an f-string.
cmd = remote_audit.replace(
    "__BENGALURU_ROOT__",
    root,
)


audit_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 5 RESULT")
print("=" * 80)

print(
    "Return code:",
    audit_result.returncode,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("Bengaluru source files modified: NO")
print("CELL 5: SUCCESS")

BENGALURU STATIC AND LEAKAGE AUDIT
Required Bengaluru files: PASS
Bengaluru Python compilation: PASS
Bengaluru configuration: PASS
Required production functions: PASS
Expected 4,900-cell constant: PASS {'EXPECTED_ROWS': 4900}
Driver template checksums: PASS
Workspace manifest checksums: PASS
Bengaluru paths and extent in code: PASS
Hyderabad leakage audit: PASS
Generalized template logic: PASS
Generalized 2020 guards: PASS 3

BENGALURU_STRUCTURAL_AUDIT_SUCCESS
No stale Bengaluru launch state: PASS

BENGALURU_STATIC_AUDIT_SUCCESS

REMOTE STDERR:


CELL 5 RESULT
Return code: 0
Spark launched: NO
HDFS modified: NO
Bengaluru source files modified: NO
CELL 5: SUCCESS


In [9]:
# ============================================================
# CELL 6: ALL-EPOCH GENERATOR SMOKE TEST
#
# Generates and compiles:
#   10 population programs
#   10 SMOD programs
#   10 built-surface programs
#
# Total: 30 generated programs
#
# This cell does not:
#   - launch Spark
#   - execute generated programs
#   - read raster data
#   - write to HDFS
#   - modify the Bengaluru workspace
#
# Generated programs are written only to /tmp and deleted.
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "bengaluru":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be bengaluru."
    )


remote_smoke_test = r'''
set -e

ROOT="__BENGALURU_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"

echo "============================================================"
echo "BENGALURU ALL-EPOCH GENERATOR SMOKE TEST"
echo "============================================================"

if [ ! -f "$DRIVER" ]; then
    echo "ERROR: Bengaluru driver is missing:"
    echo "$DRIVER"
    exit 20
fi


python3 - "$ROOT" "$DRIVER" <<'PY'
from pathlib import Path
import ast
import py_compile
import shutil
import sys
import tempfile


root = Path(sys.argv[1])
driver_path = Path(sys.argv[2])

driver_source = driver_path.read_text(
    encoding="utf-8"
)

driver_tree = ast.parse(
    driver_source
)


# ============================================================
# BUILD A DEFINITIONS-ONLY DRIVER MODULE
#
# Keep imports, constants, assignments, and functions.
# Exclude the top-level production loop so no Spark job starts.
# ============================================================

allowed_node_types = (
    ast.Import,
    ast.ImportFrom,
    ast.Assign,
    ast.AnnAssign,
    ast.FunctionDef,
    ast.AsyncFunctionDef,
    ast.ClassDef,
)

definition_nodes = [
    node
    for node in driver_tree.body
    if isinstance(
        node,
        allowed_node_types,
    )
]

definition_module = ast.Module(
    body=definition_nodes,
    type_ignores=[],
)

ast.fix_missing_locations(
    definition_module
)


namespace = {
    "__file__": str(driver_path),
    "__name__": "bengaluru_generator_smoke_test",
}

exec(
    compile(
        definition_module,
        str(driver_path),
        "exec",
    ),
    namespace,
)


# ============================================================
# REQUIRED FACTORIES
# ============================================================

required_factories = {
    "population":
        "generate_population_program",
    "smod":
        "generate_smod_program",
    "built_surface":
        "generate_built_program",
}

factories = {}

for stage, function_name in (
    required_factories.items()
):
    factory = namespace.get(
        function_name
    )

    if not callable(factory):
        raise RuntimeError(
            "Required factory is unavailable: "
            + function_name
        )

    factories[stage] = factory


# ============================================================
# TEST CONFIGURATION
# ============================================================

epochs = [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020,
]

expected_program_count = (
    len(epochs)
    * len(factories)
)

temporary_root = Path(
    tempfile.mkdtemp(
        prefix="bengaluru_generator_smoke_"
    )
)

generated_records = []


# ============================================================
# GENERATE AND COMPILE ALL 30 PROGRAMS
# ============================================================

try:
    for epoch in epochs:
        epoch_directory = (
            temporary_root
            / ("epoch_" + str(epoch))
        )

        epoch_directory.mkdir()

        for stage, factory in (
            factories.items()
        ):
            generated_object = factory(
                epoch,
                epoch_directory,
            )

            # The production factories normally return a Path.
            # Support source text as a defensive fallback.
            if isinstance(
                generated_object,
                Path,
            ):
                program_path = generated_object

            elif (
                isinstance(
                    generated_object,
                    str,
                )
                and "\n"
                not in generated_object
            ):
                program_path = Path(
                    generated_object
                )

            elif isinstance(
                generated_object,
                str,
            ):
                program_path = (
                    epoch_directory
                    / (
                        stage
                        + "_"
                        + str(epoch)
                        + ".py"
                    )
                )

                program_path.write_text(
                    generated_object,
                    encoding="utf-8",
                )

            else:
                raise RuntimeError(
                    "Unexpected factory return type "
                    "for "
                    + stage
                    + " "
                    + str(epoch)
                    + ": "
                    + type(
                        generated_object
                    ).__name__
                )


            if not program_path.is_file():
                raise RuntimeError(
                    "Factory did not create a program "
                    "for "
                    + stage
                    + " "
                    + str(epoch)
                    + ": "
                    + str(program_path)
                )


            # Compile every generated program.
            py_compile.compile(
                str(program_path),
                doraise=True,
            )


            program_source = (
                program_path.read_text(
                    encoding="utf-8"
                )
            )


            # --------------------------------------------
            # Bengaluru path validation
            # --------------------------------------------

            if (
                "city=bengaluru"
                not in program_source
            ):
                raise RuntimeError(
                    "Bengaluru HDFS path is absent from "
                    + str(program_path)
                )

            if (
                "/home/ubuntu/ghsl_multicity/"
                "bengaluru"
                not in program_source
                and stage
                in {
                    "population",
                    "smod",
                    "built_surface",
                }
            ):
                # This is informational in most generated
                # programs, but reject only if neither
                # Bengaluru workspace nor Bengaluru HDFS
                # identity appears.
                if (
                    "bengaluru"
                    not in program_source
                ):
                    raise RuntimeError(
                        "Bengaluru identity is absent from "
                        + str(program_path)
                    )


            # --------------------------------------------
            # Hyderabad leakage validation
            # --------------------------------------------

            forbidden_values = [
                "city=hyderabad",
                "/home/ubuntu/ghsl_multicity/"
                "hyderabad",
                "Hyderabad Metropolitan Region",
            ]

            remaining_forbidden = [
                value
                for value
                in forbidden_values
                if value in program_source
            ]

            if remaining_forbidden:
                raise RuntimeError(
                    "Hyderabad leakage in "
                    + str(program_path)
                    + ": "
                    + repr(
                        remaining_forbidden
                    )
                )


            # --------------------------------------------
            # Active epoch validation
            # --------------------------------------------

            expected_epoch_assignment = (
                "EPOCH = "
                + str(epoch)
            )

            compact_program_source = (
                program_source.replace(
                    " ",
                    ""
                )
            )

            compact_expected_assignment = (
                expected_epoch_assignment.replace(
                    " ",
                    ""
                )
            )

            if (
                compact_expected_assignment
                not in compact_program_source
            ):
                raise RuntimeError(
                    "Active epoch assignment is "
                    "incorrect in "
                    + str(program_path)
                    + ". Expected "
                    + expected_epoch_assignment
                )


            # --------------------------------------------
            # Expected HDFS epoch path validation
            # --------------------------------------------

            expected_epoch_path = (
                "epoch="
                + str(epoch)
                + "/"
            )

            if (
                expected_epoch_path
                not in program_source
            ):
                raise RuntimeError(
                    "Expected epoch path is absent "
                    "from "
                    + str(program_path)
                    + ": "
                    + expected_epoch_path
                )


            generated_records.append(
                {
                    "epoch": epoch,
                    "stage": stage,
                    "path": str(
                        program_path
                    ),
                }
            )

            print(
                "PASS:",
                epoch,
                stage,
                program_path.name,
            )


    # ========================================================
    # FINAL COUNTS
    # ========================================================

    if (
        len(generated_records)
        != expected_program_count
    ):
        raise RuntimeError(
            "Expected "
            + str(
                expected_program_count
            )
            + " generated programs, found "
            + str(
                len(generated_records)
            )
        )


    stage_counts = {}

    for stage in factories:
        stage_counts[stage] = sum(
            1
            for record
            in generated_records
            if record["stage"]
            == stage
        )


    expected_stage_count = len(
        epochs
    )

    for stage, count in (
        stage_counts.items()
    ):
        if count != expected_stage_count:
            raise RuntimeError(
                "Expected "
                + str(
                    expected_stage_count
                )
                + " "
                + stage
                + " programs, found "
                + str(count)
            )


    print()

    print(
        "Population programs:",
        stage_counts["population"],
    )

    print(
        "SMOD programs:",
        stage_counts["smod"],
    )

    print(
        "Built-surface programs:",
        stage_counts[
            "built_surface"
        ],
    )

    print(
        "Total generated programs:",
        len(generated_records),
    )

    print()

    print(
        "ALL_30_GENERATED_PROGRAMS_COMPILE_PASS"
    )


finally:
    # No generated smoke-test artifacts persist.
    if temporary_root.exists():
        shutil.rmtree(
            temporary_root
        )


print(
    "Temporary smoke-test directory removed: PASS"
)

print(
    "ALL_EPOCH_GENERATOR_SMOKE_TEST_SUCCESS"
)
PY


echo
echo "Spark launched: NO"
echo "HDFS modified: NO"
echo "Bengaluru workspace modified: NO"
echo "CELL_6_REMOTE_SUCCESS"
'''


# Replace a literal placeholder.
# Do not wrap the embedded Python in an outer f-string.

cmd = remote_smoke_test.replace(
    "__BENGALURU_ROOT__",
    root,
)


smoke_result = run_remote(
    cmd,
    timeout=300,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 6 RESULT")
print("=" * 80)

print(
    "Return code:",
    smoke_result.returncode,
)

print(
    "Expected generated programs:",
    30,
)

print("Spark launched: NO")
print("HDFS modified: NO")
print("Bengaluru workspace modified: NO")
print("CELL 6: SUCCESS")

BENGALURU ALL-EPOCH GENERATOR SMOKE TEST
PASS: 1975 population population_1975.py
PASS: 1975 smod smod_1975.py
PASS: 1975 built_surface built_surface_1975.py
PASS: 1980 population population_1980.py
PASS: 1980 smod smod_1980.py
PASS: 1980 built_surface built_surface_1980.py
PASS: 1985 population population_1985.py
PASS: 1985 smod smod_1985.py
PASS: 1985 built_surface built_surface_1985.py
PASS: 1990 population population_1990.py
PASS: 1990 smod smod_1990.py
PASS: 1990 built_surface built_surface_1990.py
PASS: 1995 population population_1995.py
PASS: 1995 smod smod_1995.py
PASS: 1995 built_surface built_surface_1995.py
PASS: 2000 population population_2000.py
PASS: 2000 smod smod_2000.py
PASS: 2000 built_surface built_surface_2000.py
PASS: 2005 population population_2005.py
PASS: 2005 smod smod_2005.py
PASS: 2005 built_surface built_surface_2005.py
PASS: 2010 population population_2010.py
PASS: 2010 smod smod_2010.py
PASS: 2010 built_surface built_surface_2010.py
PASS: 2015 population p

In [10]:
# ============================================================
# CELL 6A: BENGALURU SOURCE-SHARD AND TEMPLATE-ASSUMPTION PROBE
#
# Purpose:
#   - Audits all historical epochs from 1975 through 2020
#   - Counts population, SMOD, and built-surface catalog records
#   - Counts source shards intersecting Bengaluru's 70 km extent
#   - Reports the intersecting filenames, paths, raster shapes,
#     band counts, resolutions, and bounds
#   - Inspects the population and built-surface templates for
#     inherited single-shard and fixed-tile assumptions
#
# Read-only:
#   - Does not execute generated production programs
#   - Does not decode or tile raster files
#   - Does not create city outputs
#   - Does not modify the Bengaluru workspace
#   - Does not modify HDFS
#
# Important:
#   - Catalog shard counts are established here.
#   - Actual selected raster-tile counts are verified later by
#     the first controlled production execution.
# ============================================================

import base64
import hashlib
import json
import textwrap


# ============================================================
# VALIDATE NOTEBOOK STATE
# ============================================================

for required_name in [
    "ACTIVE_CITY",
    "ACTIVE_CITY_SLUG",
    "run_remote",
]:
    if required_name not in globals():
        raise RuntimeError(
            f"{required_name} is unavailable. "
            "Rerun Cells 1 and 2."
        )


if ACTIVE_CITY_SLUG != "bengaluru":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be bengaluru."
    )


root = ACTIVE_CITY[
    "remote_work_dir"
]


expected_workspace = (
    "/home/ubuntu/ghsl_multicity/bengaluru"
)


if root != expected_workspace:
    raise RuntimeError(
        "Unexpected Bengaluru workspace: "
        + root
    )


expected_extent = {
    "min_x": 7618000.0,
    "min_y": 1563000.0,
    "max_x": 7688000.0,
    "max_y": 1633000.0,
}


if (
    ACTIVE_CITY[
        "extent_mollweide"
    ]
    != expected_extent
):
    raise RuntimeError(
        "Unexpected Bengaluru extent.\n"
        "Expected: "
        + repr(expected_extent)
        + "\nActual:   "
        + repr(
            ACTIVE_CITY[
                "extent_mollweide"
            ]
        )
    )


historical_epochs = [
    1975,
    1980,
    1985,
    1990,
    1995,
    2000,
    2005,
    2010,
    2015,
    2020,
]


required_products = [
    "population",
    "smod",
    "built_surface",
]


catalog_path = (
    "hdfs://localhost:9000"
    "/data/ghsl/metadata/india/"
    "geotiff_catalog"
)


probe_config = {
    "catalog_path": catalog_path,
    "city_slug": (
        ACTIVE_CITY_SLUG
    ),
    "workspace": root,
    "extent_mollweide": (
        expected_extent
    ),
    "historical_epochs": (
        historical_epochs
    ),
    "required_products": (
        required_products
    ),
}


config_json = json.dumps(
    probe_config,
    sort_keys=True,
)


config_b64 = base64.b64encode(
    config_json.encode("utf-8")
).decode("ascii")


# ============================================================
# REMOTE SPARK CATALOG PROBE
# ============================================================

remote_probe_program = textwrap.dedent(
    r'''
    from pyspark.sql import (
        functions as F,
    )

    from sedona.spark import (
        SedonaContext,
    )

    from pathlib import Path

    import json
    import re
    import sys


    CONFIG_PATH = Path(
        sys.argv[1]
    )


    if not CONFIG_PATH.is_file():
        raise FileNotFoundError(
            CONFIG_PATH
        )


    config = json.loads(
        CONFIG_PATH.read_text(
            encoding="utf-8"
        )
    )


    CATALOG_PATH = config[
        "catalog_path"
    ]

    CITY_SLUG = config[
        "city_slug"
    ]

    WORKSPACE = Path(
        config[
            "workspace"
        ]
    )

    EXTENT = config[
        "extent_mollweide"
    ]

    HISTORICAL_EPOCHS = [
        int(epoch)
        for epoch
        in config[
            "historical_epochs"
        ]
    ]

    REQUIRED_PRODUCTS = config[
        "required_products"
    ]


    POPULATION_TEMPLATE = (
        WORKSPACE
        / "templates"
        / "population_epoch_template.py"
    )

    SMOD_TEMPLATE = (
        WORKSPACE
        / "templates"
        / "smod_epoch_template.py"
    )

    BUILT_TEMPLATE = (
        WORKSPACE
        / "templates"
        / "built_surface_epoch_template.py"
    )


    for required_file in [
        POPULATION_TEMPLATE,
        SMOD_TEMPLATE,
        BUILT_TEMPLATE,
    ]:
        if not required_file.is_file():
            raise FileNotFoundError(
                required_file
            )


    # ========================================================
    # TEMPLATE-ASSUMPTION INSPECTION
    # ========================================================

    population_source = (
        POPULATION_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    smod_source = (
        SMOD_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )

    built_source = (
        BUILT_TEMPLATE.read_text(
            encoding="utf-8"
        )
    )


    def inspect_template(
        source,
        template_name,
    ):
        exact_one_shard_fragments = [
            fragment
            for fragment
            in [
                "selected_source_count != 1",
                "selected_source_count == 1",
                "Expected exactly one",
                "exactly one source shard",
            ]
            if fragment in source
        ]

        first_source_fragments = [
            fragment
            for fragment
            in [
                "selected_sources_df.first()",
                ".first()",
            ]
            if fragment in source
        ]

        single_path_load_fragments = [
            fragment
            for fragment
            in [
                ".load(source_path)",
                ".load(selected_source_path)",
            ]
            if fragment in source
        ]

        multi_path_load_fragments = [
            fragment
            for fragment
            in [
                ".load(selected_source_paths)",
                ".load(source_paths)",
            ]
            if fragment in source
        ]

        fixed_tile_constants = {}

        for match in re.finditer(
            r"^([A-Z][A-Z0-9_]*)\s*=\s*(\d+)\s*$",
            source,
            flags=re.MULTILINE,
        ):
            constant_name = (
                match.group(1)
            )

            constant_value = int(
                match.group(2)
            )

            if (
                "TILE"
                in constant_name
            ):
                fixed_tile_constants[
                    constant_name
                ] = constant_value

        return {
            "template_name": (
                template_name
            ),
            "exact_one_shard_fragments": (
                exact_one_shard_fragments
            ),
            "first_source_fragments": (
                first_source_fragments
            ),
            "single_path_load_fragments": (
                single_path_load_fragments
            ),
            "multi_path_load_fragments": (
                multi_path_load_fragments
            ),
            "fixed_tile_constants": (
                fixed_tile_constants
            ),
            "contains_selected_source_count": (
                "selected_source_count"
                in source
            ),
            "contains_selected_source_paths": (
                "selected_source_paths"
                in source
            ),
        }


    template_reports = {
        "population": inspect_template(
            population_source,
            "population_epoch_template.py",
        ),
        "smod": inspect_template(
            smod_source,
            "smod_epoch_template.py",
        ),
        "built_surface": inspect_template(
            built_source,
            "built_surface_epoch_template.py",
        ),
    }


    # ========================================================
    # CREATE SPARK + SEDONA CONTEXT
    # ========================================================

    spark_config = (
        SedonaContext.builder()
        .appName(
            "GHSL-Bengaluru-"
            "Historical-Source-Shard-Probe"
        )
        .getOrCreate()
    )

    spark = SedonaContext.create(
        spark_config
    )

    spark.sparkContext.setLogLevel(
        "WARN"
    )


    try:
        # ====================================================
        # READ AND VALIDATE CATALOG
        # ====================================================

        catalog_df = (
            spark.read
            .parquet(
                CATALOG_PATH
            )
        )


        required_catalog_columns = {
            "product",
            "temporal_value",
            "filename",
            "path",
            "fileSize",
            "width",
            "height",
            "numBands",
            "resolution_x",
            "resolution_y",
            "min_x",
            "min_y",
            "max_x",
            "max_y",
        }


        missing_catalog_columns = sorted(
            required_catalog_columns
            - set(
                catalog_df.columns
            )
        )


        if missing_catalog_columns:
            raise RuntimeError(
                "The GeoTIFF catalog is missing "
                "required columns: "
                + repr(
                    missing_catalog_columns
                )
            )


        total_catalog_records = (
            catalog_df.count()
        )


        if total_catalog_records <= 0:
            raise RuntimeError(
                "The GeoTIFF catalog is empty."
            )


        catalog_reports = []


        # ====================================================
        # AUDIT ALL 30 PRODUCT-EPOCH COMBINATIONS
        # ====================================================

        for product in REQUIRED_PRODUCTS:
            for epoch in HISTORICAL_EPOCHS:
                product_epoch_df = (
                    catalog_df
                    .filter(
                        F.lower(
                            F.trim(
                                F.col(
                                    "product"
                                )
                            )
                        )
                        == F.lit(
                            product
                        )
                    )
                    .filter(
                        F.col(
                            "temporal_value"
                        )
                        .cast("int")
                        == F.lit(
                            epoch
                        )
                    )
                )


                catalog_record_count = (
                    product_epoch_df.count()
                )


                usable_df = (
                    product_epoch_df
                    .filter(
                        F.col(
                            "path"
                        ).isNotNull()
                    )
                    .filter(
                        F.col(
                            "filename"
                        ).isNotNull()
                    )
                    .filter(
                        F.col(
                            "min_x"
                        ).isNotNull()
                    )
                    .filter(
                        F.col(
                            "min_y"
                        ).isNotNull()
                    )
                    .filter(
                        F.col(
                            "max_x"
                        ).isNotNull()
                    )
                    .filter(
                        F.col(
                            "max_y"
                        ).isNotNull()
                    )
                )


                usable_record_count = (
                    usable_df.count()
                )


                usable_df.createOrReplaceTempView(
                    "probe_product_epoch"
                )


                intersecting_df = spark.sql(
                    f"""
                    SELECT
                        filename,
                        path,
                        fileSize,
                        width,
                        height,
                        numBands,
                        resolution_x,
                        resolution_y,
                        min_x,
                        min_y,
                        max_x,
                        max_y

                    FROM probe_product_epoch

                    WHERE ST_Intersects(
                        ST_PolygonFromEnvelope(
                            min_x,
                            min_y,
                            max_x,
                            max_y
                        ),
                        ST_PolygonFromEnvelope(
                            {float(EXTENT["min_x"])},
                            {float(EXTENT["min_y"])},
                            {float(EXTENT["max_x"])},
                            {float(EXTENT["max_y"])}
                        )
                    )

                    ORDER BY filename
                    """
                )


                intersecting_records = [
                    row.asDict(
                        recursive=True
                    )
                    for row
                    in intersecting_df.collect()
                ]


                intersecting_shard_count = len(
                    intersecting_records
                )


                distinct_intersecting_paths = len(
                    {
                        record[
                            "path"
                        ]
                        for record
                        in intersecting_records
                        if record[
                            "path"
                        ]
                        is not None
                    }
                )


                if (
                    intersecting_shard_count
                    != distinct_intersecting_paths
                ):
                    raise RuntimeError(
                        "Duplicate intersecting source paths "
                        "for "
                        + product
                        + " "
                        + str(epoch)
                        + "."
                    )


                catalog_reports.append(
                    {
                        "product": product,
                        "epoch": epoch,
                        "catalog_record_count": (
                            catalog_record_count
                        ),
                        "usable_record_count": (
                            usable_record_count
                        ),
                        "intersecting_shard_count": (
                            intersecting_shard_count
                        ),
                        "distinct_intersecting_paths": (
                            distinct_intersecting_paths
                        ),
                        "intersecting_records": (
                            intersecting_records
                        ),
                    }
                )


                print(
                    "PROBE:",
                    product,
                    epoch,
                    "catalog_records="
                    + str(
                        catalog_record_count
                    ),
                    "intersecting_shards="
                    + str(
                        intersecting_shard_count
                    ),
                )


        # ====================================================
        # SHARD-COUNT STABILITY BY PRODUCT
        # ====================================================

        shard_counts_by_product = {}


        for product in REQUIRED_PRODUCTS:
            product_reports = [
                report
                for report
                in catalog_reports
                if (
                    report["product"]
                    == product
                )
            ]

            counts_by_epoch = {
                str(
                    report[
                        "epoch"
                    ]
                ): report[
                    "intersecting_shard_count"
                ]
                for report
                in product_reports
            }

            unique_counts = sorted(
                set(
                    counts_by_epoch.values()
                )
            )

            shard_counts_by_product[
                product
            ] = {
                "counts_by_epoch": (
                    counts_by_epoch
                ),
                "unique_counts": (
                    unique_counts
                ),
                "stable_across_epochs": (
                    len(
                        unique_counts
                    )
                    == 1
                ),
            }


        # ====================================================
        # DETERMINE TEMPLATE COMPATIBILITY
        # ====================================================

        compatibility_reports = {}


        for product in [
            "population",
            "built_surface",
        ]:
            template_report = (
                template_reports[
                    product
                ]
            )

            shard_report = (
                shard_counts_by_product[
                    product
                ]
            )

            maximum_shards = max(
                shard_report[
                    "unique_counts"
                ]
            )

            minimum_shards = min(
                shard_report[
                    "unique_counts"
                ]
            )

            exact_one_assumption = bool(
                template_report[
                    "exact_one_shard_fragments"
                ]
                or template_report[
                    "single_path_load_fragments"
                ]
            )

            multi_path_support = bool(
                template_report[
                    "multi_path_load_fragments"
                ]
            )

            patch_required = (
                maximum_shards > 1
                and exact_one_assumption
                and not multi_path_support
            )

            source_missing = (
                minimum_shards < 1
            )

            compatibility_reports[
                product
            ] = {
                "minimum_intersecting_shards": (
                    minimum_shards
                ),
                "maximum_intersecting_shards": (
                    maximum_shards
                ),
                "exact_one_shard_assumption": (
                    exact_one_assumption
                ),
                "multi_path_support": (
                    multi_path_support
                ),
                "patch_required": (
                    patch_required
                ),
                "source_missing": (
                    source_missing
                ),
            }


        missing_product_epochs = [
            {
                "product": report[
                    "product"
                ],
                "epoch": report[
                    "epoch"
                ],
            }
            for report
            in catalog_reports
            if (
                report[
                    "catalog_record_count"
                ]
                < 1
                or report[
                    "usable_record_count"
                ]
                < 1
                or report[
                    "intersecting_shard_count"
                ]
                < 1
            )
        ]


        output = {
            "city_slug": CITY_SLUG,
            "workspace": str(
                WORKSPACE
            ),
            "catalog_path": (
                CATALOG_PATH
            ),
            "total_catalog_records": (
                total_catalog_records
            ),
            "extent_mollweide": (
                EXTENT
            ),
            "historical_epochs": (
                HISTORICAL_EPOCHS
            ),
            "required_products": (
                REQUIRED_PRODUCTS
            ),
            "template_reports": (
                template_reports
            ),
            "catalog_reports": (
                catalog_reports
            ),
            "shard_counts_by_product": (
                shard_counts_by_product
            ),
            "compatibility_reports": (
                compatibility_reports
            ),
            "missing_product_epochs": (
                missing_product_epochs
            ),
            "all_historical_sources_available": (
                len(
                    missing_product_epochs
                )
                == 0
            ),
        }


        print(
            "BENGALURU_SHARD_PROBE_JSON_BEGIN"
        )

        print(
            json.dumps(
                output,
                sort_keys=True,
            )
        )

        print(
            "BENGALURU_SHARD_PROBE_JSON_END"
        )

        print(
            "BENGALURU_HISTORICAL_"
            "SOURCE_SHARD_PROBE_COMPLETE"
        )


    finally:
        spark.stop()
    '''
).strip() + "\n"


# ============================================================
# VALIDATE AND ENCODE REMOTE PROGRAM
# ============================================================

compile(
    remote_probe_program,
    "probe_bengaluru_historical_shards.py",
    "exec",
)


probe_program_sha256 = hashlib.sha256(
    remote_probe_program.encode(
        "utf-8"
    )
).hexdigest()


probe_program_b64 = base64.b64encode(
    remote_probe_program.encode(
        "utf-8"
    )
).decode("ascii")


print(
    "LOCAL BENGALURU SHARD PROBE "
    "COMPILE: PASS"
)

print(
    "Expected probe SHA-256:",
    probe_program_sha256,
)


# ============================================================
# TRANSFER AND EXECUTE
# ============================================================

remote_command = f"""
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}"

SCRIPT="/tmp/probe_bengaluru_historical_shards.py"
CONFIG="/tmp/bengaluru_historical_shard_config.json"

EXPECTED_SCRIPT_SHA256="{probe_program_sha256}"

echo "{probe_program_b64}" \
| base64 -d \
> "${{SCRIPT}}"

echo "{config_b64}" \
| base64 -d \
> "${{CONFIG}}"

ACTUAL_SCRIPT_SHA256="$(
    sha256sum "${{SCRIPT}}" \
    | awk '{{print $1}}'
)"

echo "Remote probe SHA-256: ${{ACTUAL_SCRIPT_SHA256}}"

if [ "${{ACTUAL_SCRIPT_SHA256}}" != "${{EXPECTED_SCRIPT_SHA256}}" ]; then
    echo "ERROR: Probe-script checksum mismatch."
    exit 20
fi

echo "REMOTE PROBE CHECKSUM: PASS"

python3 -m py_compile "${{SCRIPT}}"

echo "REMOTE PROBE COMPILE: PASS"

spark-submit \
    --master local[1] \
    --driver-memory 3g \
    --conf spark.ui.enabled=false \
    --conf spark.sql.shuffle.partitions=2 \
    --conf spark.default.parallelism=2 \
    --conf spark.sql.adaptive.enabled=false \
    --conf spark.sql.codegen.wholeStage=false \
    --conf spark.driver.maxResultSize=256m \
    --packages \
org.apache.sedona:sedona-spark-3.5_2.12:1.9.1,org.datasyslab:geotools-wrapper:1.9.1-33.5 \
    --repositories \
https://artifacts.unidata.ucar.edu/repository/unidata-all \
    "${{SCRIPT}}" \
    "${{CONFIG}}"

rm -f \
    "${{SCRIPT}}" \
    "${{CONFIG}}"

echo "Temporary probe files removed: PASS"
echo "Bengaluru workspace modified: NO"
echo "HDFS modified: NO"
echo "BENGALURU_SHARD_PROBE_REMOTE_SUCCESS"
"""


probe_result = run_remote(
    remote_command,
    timeout=1200,
    show_output=False,
)


# ============================================================
# PARSE REMOTE JSON WITHOUT MULTILINE REGEX
# ============================================================

begin_marker = (
    "BENGALURU_SHARD_PROBE_JSON_BEGIN"
)

end_marker = (
    "BENGALURU_SHARD_PROBE_JSON_END"
)


begin_index = (
    probe_result.stdout.find(
        begin_marker
    )
)

end_index = (
    probe_result.stdout.find(
        end_marker
    )
)


if (
    begin_index < 0
    or end_index < 0
    or end_index <= begin_index
):
    print(
        probe_result.stdout[
            -12000:
        ]
    )

    if probe_result.stderr:
        print(
            "REMOTE STDERR:"
        )

        print(
            probe_result.stderr[
                -4000:
            ]
        )

    raise RuntimeError(
        "Could not locate the Bengaluru "
        "shard-probe JSON markers."
    )


json_start = (
    begin_index
    + len(
        begin_marker
    )
)


probe_json = (
    probe_result.stdout[
        json_start:
        end_index
    ]
    .strip()
)


if not probe_json:
    raise RuntimeError(
        "The Bengaluru shard-probe JSON "
        "payload is empty."
    )


try:
    BENGALURU_SHARD_PROBE = (
        json.loads(
            probe_json
        )
    )

except json.JSONDecodeError as error:
    print(
        probe_json[
            :12000
        ]
    )

    raise RuntimeError(
        "Could not parse the Bengaluru "
        "shard-probe JSON."
    ) from error


# ============================================================
# VALIDATE REPORT STRUCTURE
# ============================================================

catalog_reports = (
    BENGALURU_SHARD_PROBE.get(
        "catalog_reports",
        [],
    )
)


expected_report_count = (
    len(historical_epochs)
    * len(required_products)
)


if (
    len(catalog_reports)
    != expected_report_count
):
    raise RuntimeError(
        "Expected "
        + str(
            expected_report_count
        )
        + " product-epoch reports, found "
        + str(
            len(catalog_reports)
        )
    )


# ============================================================
# REPORT
# ============================================================

print()
print("=" * 80)
print(
    "BENGALURU HISTORICAL SOURCE-SHARD PROBE"
)
print("=" * 80)

print(
    "Catalog:",
    BENGALURU_SHARD_PROBE[
        "catalog_path"
    ],
)

print(
    "Catalog records:",
    BENGALURU_SHARD_PROBE[
        "total_catalog_records"
    ],
)

print(
    "Bengaluru extent:",
    BENGALURU_SHARD_PROBE[
        "extent_mollweide"
    ],
)


shard_counts_by_product = (
    BENGALURU_SHARD_PROBE[
        "shard_counts_by_product"
    ]
)


for product in required_products:
    product_report = (
        shard_counts_by_product[
            product
        ]
    )

    print()

    print(
        product.upper()
    )

    print(
        "  Counts by epoch:",
        product_report[
            "counts_by_epoch"
        ],
    )

    print(
        "  Unique shard counts:",
        product_report[
            "unique_counts"
        ],
    )

    print(
        "  Stable across epochs:",
        product_report[
            "stable_across_epochs"
        ],
    )


print()

print(
    "Template compatibility:"
)


compatibility_reports = (
    BENGALURU_SHARD_PROBE[
        "compatibility_reports"
    ]
)


for product in [
    "population",
    "built_surface",
]:
    report = (
        compatibility_reports[
            product
        ]
    )

    print()

    print(
        " ",
        product,
    )

    print(
        "    minimum shards:",
        report[
            "minimum_intersecting_shards"
        ],
    )

    print(
        "    maximum shards:",
        report[
            "maximum_intersecting_shards"
        ],
    )

    print(
        "    exact-one assumption:",
        report[
            "exact_one_shard_assumption"
        ],
    )

    print(
        "    multi-path support:",
        report[
            "multi_path_support"
        ],
    )

    print(
        "    patch required:",
        report[
            "patch_required"
        ],
    )

    print(
        "    source missing:",
        report[
            "source_missing"
        ],
    )


print()

print(
    "All historical sources available:",
    BENGALURU_SHARD_PROBE[
        "all_historical_sources_available"
    ],
)

print(
    "Missing product-epochs:",
    BENGALURU_SHARD_PROBE[
        "missing_product_epochs"
    ],
)

print(
    "Actual raster decoding performed: NO"
)

print(
    "Actual tile generation performed: NO"
)

print(
    "Bengaluru workspace modified: NO"
)

print(
    "HDFS modified: NO"
)

print(
    "Temporary probe files removed: PASS"
)


population_patch_required = (
    compatibility_reports[
        "population"
    ][
        "patch_required"
    ]
)


built_patch_required = (
    compatibility_reports[
        "built_surface"
    ][
        "patch_required"
    ]
)


if not BENGALURU_SHARD_PROBE[
    "all_historical_sources_available"
]:
    print(
        "BENGALURU_HISTORICAL_SOURCE_GAP_FOUND"
    )

elif (
    population_patch_required
    or built_patch_required
):
    print(
        "BENGALURU_MULTI_SHARD_PATCH_REQUIRED"
    )

else:
    print(
        "BENGALURU_SOURCE_SHARD_PREFLIGHT_SUCCESS"
    )


print(
    "CELL 6A: SUCCESS"
)

LOCAL BENGALURU SHARD PROBE COMPILE: PASS
Expected probe SHA-256: 2c6ca7cbeb0693aeeb3ca5ae8d53fc1e51e0e5f8c97f34255d5564d607f4585c

BENGALURU HISTORICAL SOURCE-SHARD PROBE
Catalog: hdfs://localhost:9000/data/ghsl/metadata/india/geotiff_catalog
Catalog records: 325
Bengaluru extent: {'max_x': 7688000.0, 'max_y': 1633000.0, 'min_x': 7618000.0, 'min_y': 1563000.0}

POPULATION
  Counts by epoch: {'1975': 1, '1980': 1, '1985': 1, '1990': 1, '1995': 1, '2000': 1, '2005': 1, '2010': 1, '2015': 1, '2020': 1}
  Unique shard counts: [1]
  Stable across epochs: True

SMOD
  Counts by epoch: {'1975': 1, '1980': 1, '1985': 1, '1990': 1, '1995': 1, '2000': 1, '2005': 1, '2010': 1, '2015': 1, '2020': 1}
  Unique shard counts: [1]
  Stable across epochs: True

BUILT_SURFACE
  Counts by epoch: {'1975': 1, '1980': 1, '1985': 1, '1990': 1, '1995': 1, '2000': 1, '2005': 1, '2010': 1, '2015': 1, '2020': 1}
  Unique shard counts: [1]
  Stable across epochs: True

Template compatibility:

  population
    mi

In [11]:
# ============================================================
# CELL 7: DETACHED BENGALURU PRODUCTION LAUNCH
#
# Launches the full 1975-2020 Bengaluru production pipeline.
#
# Safety:
#   - Refuses a duplicate active launch
#   - Loads Hadoop/Sedona environment
#   - Verifies HDFS daemons and safe mode
#   - Verifies Bengaluru driver and template checksums
#   - Writes PID before returning
#   - Detects immediate startup failure
#   - Leaves the job detached on EC2
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]

if ACTIVE_CITY_SLUG != "bengaluru":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be bengaluru."
    )


remote_launch = r'''
set -e

source /home/ubuntu/ghsl-sedona-env.sh

export HADOOP_HOME=/usr/local/hadoop
export PATH="${HADOOP_HOME}/bin:${HADOOP_HOME}/sbin:${PATH}"

ROOT="__BENGALURU_ROOT__"
DRIVER="$ROOT/batch/run_city_epochs.py"
PID_FILE="$ROOT/batch/city.pid"
LOG_FILE="$ROOT/batch/city.log"


echo "============================================================"
echo "BENGALURU PRODUCTION LAUNCH"
echo "============================================================"


# ------------------------------------------------------------
# Verify HDFS services.
# ------------------------------------------------------------

for daemon in \
    NameNode \
    DataNode \
    SecondaryNameNode

do
    jps \
        | awk '{print $2}' \
        | grep -qx "$daemon"
done

hdfs dfsadmin -safemode get \
    | grep -qi "Safe mode is OFF"

echo "HDFS services: PASS"


# ------------------------------------------------------------
# Verify Bengaluru production files.
# ------------------------------------------------------------

if [ ! -f "$DRIVER" ]; then
    echo "ERROR: Bengaluru driver is missing."
    exit 20
fi

python3 -m py_compile "$DRIVER"

for template in "$ROOT"/templates/*.py; do
    python3 -m py_compile "$template"
done

echo "Bengaluru Python compilation: PASS"


# ------------------------------------------------------------
# Refuse duplicate launch.
# ------------------------------------------------------------

if [ -f "$PID_FILE" ]; then
    OLD_PID="$(cat "$PID_FILE")"

    if kill -0 "$OLD_PID" 2>/dev/null; then
        echo "ERROR: Bengaluru job is already running."
        echo "Active PID: $OLD_PID"
        exit 30
    fi

    echo "Removing stale PID file:"
    echo "$PID_FILE"

    rm -f "$PID_FILE"
fi


# ------------------------------------------------------------
# Preserve an old log if one unexpectedly exists.
# ------------------------------------------------------------

if [ -f "$LOG_FILE" ]; then
    LOG_BACKUP="$LOG_FILE.before_launch.$(
        date -u +%Y%m%dT%H%M%SZ
    )"

    mv "$LOG_FILE" "$LOG_BACKUP"

    echo "Previous log moved to:"
    echo "$LOG_BACKUP"
fi


# ------------------------------------------------------------
# Launch detached production driver.
# ------------------------------------------------------------

nohup python3 "$DRIVER" \
    >"$LOG_FILE" \
    2>&1 \
    </dev/null &

NEW_PID="$!"

echo "$NEW_PID" > "$PID_FILE"

echo "New PID: $NEW_PID"
echo "Log: $LOG_FILE"


# ------------------------------------------------------------
# Allow startup checks to run.
# ------------------------------------------------------------

sleep 5


# ------------------------------------------------------------
# Confirm process survived startup.
# ------------------------------------------------------------

if ! kill -0 "$NEW_PID" 2>/dev/null; then
    echo
    echo "ERROR: Bengaluru driver exited during startup."

    echo
    echo "Latest 100 log lines:"

    if [ -f "$LOG_FILE" ]; then
        tail -n 100 "$LOG_FILE"
    fi

    exit 31
fi


echo

ps -p "$NEW_PID" \
    -o pid,etime,%cpu,%mem,cmd

echo
echo "DETACHED_LAUNCH_SUCCESS"
'''


cmd = remote_launch.replace(
    "__BENGALURU_ROOT__",
    root,
)


launch_result = run_remote(
    cmd,
    timeout=90,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 7 RESULT")
print("=" * 80)

print(
    "Return code:",
    launch_result.returncode,
)

print("Safe to close Colab.")
print("Monitor with Cell 8.")
print("CELL 7: SUCCESS")

BENGALURU PRODUCTION LAUNCH
HDFS services: PASS
Bengaluru Python compilation: PASS
New PID: 65459
Log: /home/ubuntu/ghsl_multicity/bengaluru/batch/city.log

    PID     ELAPSED %CPU %MEM CMD
  65459       00:05  0.3  0.2 python3 /home/ubuntu/ghsl_multicity/bengaluru/batch/run_city_epochs.py

DETACHED_LAUNCH_SUCCESS

REMOTE STDERR:


CELL 7 RESULT
Return code: 0
Safe to close Colab.
Monitor with Cell 8.
CELL 7: SUCCESS


In [18]:
# ============================================================
# CELL 8: STABLE BENGALURU MONITOR WITH CLEAR FINAL STATUS
#
# Read-only:
#   - Does not launch a job
#   - Does not stop a job
#   - Does not modify HDFS
# ============================================================

root = ACTIVE_CITY["remote_work_dir"]
city_name = ACTIVE_CITY["city_name"]

if ACTIVE_CITY_SLUG != "bengaluru":
    raise RuntimeError(
        "ACTIVE_CITY_SLUG must be bengaluru."
    )


remote_monitor = r'''
set +e

source /home/ubuntu/ghsl-sedona-env.sh 2>/dev/null

export HADOOP_HOME=/usr/local/hadoop
export PATH="$HADOOP_HOME/bin:$HADOOP_HOME/sbin:$PATH"

ROOT="__BENGALURU_ROOT__"
CITY_NAME="__CITY_NAME__"

PID_FILE="$ROOT/batch/city.pid"
LOG_FILE="$ROOT/batch/city.log"
MANIFEST="$ROOT/batch/batch_manifest.tsv"


get_job_status() {
    RECORDED_PID="N/A"
    JOB_STATUS="UNKNOWN"

    if [ -f "$PID_FILE" ]; then
        RECORDED_PID="$(cat "$PID_FILE")"

        if kill -0 "$RECORDED_PID" 2>/dev/null; then
            JOB_STATUS="RUNNING"
        else
            JOB_STATUS="STOPPED"
        fi
    fi
}


# ============================================================
# INITIAL STATUS
# ============================================================

get_job_status

echo "============================================================"
echo "BENGALURU JOB STATUS"
echo "============================================================"

echo "City: $CITY_NAME"
echo "Recorded PID: $RECORDED_PID"
echo "STATUS=$JOB_STATUS"

if [ "$JOB_STATUS" = "RUNNING" ]; then
    ps -p "$RECORDED_PID" \
        -o pid,etime,%cpu,%mem,cmd
fi


# ============================================================
# LATEST LOG
# ============================================================

echo
echo "============================================================"
echo "LATEST 150 LOG LINES"
echo "============================================================"

if [ -f "$LOG_FILE" ]; then
    tail -n 150 "$LOG_FILE"
else
    echo "Log file is missing."
fi


# ============================================================
# CURRENT MANIFEST
# ============================================================

echo
echo "============================================================"
echo "CURRENT BENGALURU MANIFEST"
echo "============================================================"

if [ -f "$MANIFEST" ]; then
    cat "$MANIFEST"
else
    echo "Manifest has not been created."
fi


# ============================================================
# SIMPLE PROGRESS COUNTS
# ============================================================

echo
echo "============================================================"
echo "CURRENT PROGRESS SUMMARY"
echo "============================================================"

COMPLETED_STAGES=0
COMPLETED_EPOCHS=0

if [ -f "$MANIFEST" ]; then
    COMPLETED_STAGES="$(
        grep -E \
            $'\t(built_and_validated|existing_valid_skipped)\t' \
            "$MANIFEST" \
            | wc -l \
            | tr -d ' '
    )"

    for epoch in \
        1975 1980 1985 1990 1995 \
        2000 2005 2010 2015 2020
    do
        EPOCH_RECORDS="$(
            awk -F '\t' \
                -v target_epoch="$epoch" \
                '$1 == target_epoch {print}' \
                "$MANIFEST" \
                | grep -E \
                    $'\t(built_and_validated|existing_valid_skipped)\t' \
                | wc -l \
                | tr -d ' '
        )"

        if [ "$EPOCH_RECORDS" = "4" ]; then
            COMPLETED_EPOCHS="$(
                expr "$COMPLETED_EPOCHS" + 1
            )"
        fi
    done
fi

echo "Completed stage records: $COMPLETED_STAGES / 40"
echo "Fully completed epochs: $COMPLETED_EPOCHS / 10"


# ============================================================
# ACTIVE SPARK / JAVA PROCESSES
# ============================================================

echo
echo "============================================================"
echo "ACTIVE SPARK / JAVA PROCESSES"
echo "============================================================"

jps


# ============================================================
# FINAL STATUS
# ============================================================

get_job_status

COMPLETION_RESULT="UNKNOWN"
FINAL_MESSAGE="BENGALURU JOB STATUS UNKNOWN"

if [ "$JOB_STATUS" = "RUNNING" ]; then
    COMPLETION_RESULT="IN_PROGRESS"
    FINAL_MESSAGE="BENGALURU JOB IS STILL RUNNING"
fi

if [ "$JOB_STATUS" = "STOPPED" ]; then
    if [ -f "$LOG_FILE" ]; then
        if grep -q "BATCH PIPELINE: SUCCESS" "$LOG_FILE"; then
            COMPLETION_RESULT="SUCCESS"
            FINAL_MESSAGE="BENGALURU JOB COMPLETED SUCCESSFULLY"
        elif grep -q "Traceback (most recent call last)" "$LOG_FILE"; then
            COMPLETION_RESULT="FAILED"
            FINAL_MESSAGE="BENGALURU JOB STOPPED WITH FAILURE"
        else
            COMPLETION_RESULT="UNKNOWN"
            FINAL_MESSAGE="BENGALURU JOB STOPPED, RESULT UNKNOWN"
        fi
    fi
fi


# ============================================================
# FINAL BLOCK
# ============================================================

echo
echo
echo "############################################################"
echo "############################################################"
echo
echo "              FINAL BENGALURU PIPELINE STATUS"
echo
echo "City: $CITY_NAME"
echo "PID: $RECORDED_PID"
echo "FINAL_STATUS=$JOB_STATUS"
echo "COMPLETION_RESULT=$COMPLETION_RESULT"
echo "COMPLETED_STAGES=$COMPLETED_STAGES/40"
echo "COMPLETED_EPOCHS=$COMPLETED_EPOCHS/10"
echo
echo ">>> $FINAL_MESSAGE <<<"
echo

if [ "$JOB_STATUS" = "RUNNING" ]; then
    ps -p "$RECORDED_PID" \
        -o pid,etime,%cpu,%mem,cmd

    echo
    echo "Action: Let the EC2 job continue."
    echo "Do not rerun Cell 7."
fi

if [ "$COMPLETION_RESULT" = "SUCCESS" ]; then
    echo "Action: Run Cell 9 final audit."
fi

if [ "$COMPLETION_RESULT" = "FAILED" ]; then
    echo "Latest failure markers:"

    grep -n \
        -E \
        "Traceback|RuntimeError|AnalysisException|Return code: 1" \
        "$LOG_FILE" \
        | tail -n 20

    echo
    echo "Action: Do not rerun Cell 7."
fi

echo
echo "############################################################"
echo "############################################################"

exit 0
'''


cmd = (
    remote_monitor
    .replace(
        "__BENGALURU_ROOT__",
        root,
    )
    .replace(
        "__CITY_NAME__",
        city_name,
    )
)


monitor_result = run_remote(
    cmd,
    timeout=180,
    show_output=True,
)


print()
print("=" * 80)
print("CELL 8 RESULT")
print("=" * 80)

print(
    "Monitor return code:",
    monitor_result.returncode,
)

print("No new job was launched.")
print("No running process was stopped.")
print("No HDFS data was modified.")

BENGALURU JOB STATUS
City: Bengaluru Metropolitan Region
Recorded PID: 65459
STATUS=STOPPED

LATEST 150 LOG LINES
VALIDATE NEW ENRICHED 2015
Command: spark-submit --master local[1] --driver-memory 1g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=2 --conf spark.default.parallelism=2 --conf spark.sql.adaptive.enabled=false --conf spark.sql.codegen.wholeStage=false --conf spark.driver.maxResultSize=256m /home/ubuntu/ghsl_multicity/bengaluru/programs/epoch_2015/validate_enriched_2015.py


Return code: 0
Elapsed seconds: 12.23
enriched 2015: BUILT AND VALIDATED

EPOCH 2015: COMPLETE
Epoch elapsed minutes: 10.33

################################################################################
EPOCH 2020 (10/10)
################################################################################

RUN POPULATION 2020
Command: spark-submit --master local[1] --driver-memory 5g --conf spark.ui.enabled=false --conf spark.sql.shuffle.partitions=4 --conf spark.default.parallelism=2 -

In [19]:
# CELL 9: FINAL 40-PATH AUDIT
root=ACTIVE_CITY['remote_work_dir']; slug=ACTIVE_CITY_SLUG
cmd=f"""set -e
source /home/ubuntu/ghsl-sedona-env.sh
export HADOOP_HOME=/usr/local/hadoop; export PATH=${{HADOOP_HOME}}/bin:${{HADOOP_HOME}}/sbin:${{PATH}}
grep -q 'BATCH PIPELINE: SUCCESS' {root}/batch/city.log
n=0
for e in 1975 1980 1985 1990 1995 2000 2005 2010 2015 2020; do
 for s in population smod built_surface; do hdfs dfs -test -e /data/ghsl/curated/cities/city={slug}/product=$s/epoch=$e/grid_1km; n=$((n+1)); done
 hdfs dfs -test -e /data/ghsl/curated/cities/city={slug}/enriched_grid_year/epoch=$e; n=$((n+1)); echo Epoch $e: 4/4 paths PASS
done
test $n -eq 40; echo ALL_40_PATHS_PASS; echo CITY_COMPLETE
"""
run_remote(cmd,timeout=300); print('CELL 9: SUCCESS')


Epoch 1975: 4/4 paths PASS
Epoch 1980: 4/4 paths PASS
Epoch 1985: 4/4 paths PASS
Epoch 1990: 4/4 paths PASS
Epoch 1995: 4/4 paths PASS
Epoch 2000: 4/4 paths PASS
Epoch 2005: 4/4 paths PASS
Epoch 2010: 4/4 paths PASS
Epoch 2015: 4/4 paths PASS
Epoch 2020: 4/4 paths PASS
ALL_40_PATHS_PASS
CITY_COMPLETE

REMOTE STDERR:

CELL 9: SUCCESS


The standardized modeling dataset is now available under:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/cities/city=bengaluru/

The legacy pilot remains unchanged under:

Plain Text
hdfs://localhost:9000/data/ghsl/curated/pilots/bengaluru/